In [34]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

[1] "competitions"

Phase 1

In [35]:
# =============================================================================
# PHASE 1: LOAD, CLEAN EXPOSURE, MATCH CLAIMS, JOIN
#
# Reproducible and dataset-agnostic
# values are hard-coded. To run on a different dataset (e.g. Week 2), change
# only the SETTINGS in Cell 1.
#

# Output (in /kaggle/working/<run_folder>):
#   train_<line>.parquet, test_<line>.parquet   joined policy rows (one per row and
#                                               calendar year; orig_row_id = Kaggle row_id)
#   claims_matched.parquet                      claims with the row_id they belong to
#   claims_double_matched.parquet               claims that fell in two rows (resolved)
#   claims_unmatched.parquet                    claims outside every row (excluded)
#   phase1_summary.csv                          counts for the write-up
#   session_info.txt                            R and package versions used
# =============================================================================


# %% Cell 1: Settings and packages ============================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
data_dir      <- NULL      # folder with the raw .parquet files; NULL = find it automatically
run_folder    <- "prep"    # output folder name; use e.g. "prep_week2" for Week 2
lines         <- c("motor", "buildings", "contents")
days_per_year <- 365       # data convention: every policy year counts as 365 days
leap_ends_dec30 <- TRUE    # data convention: in leap years cover stops on 30 Dec (31 Dec not covered)
drop_cols     <- c("renewal_decision",                       # training only: leakage
                   "surname", "email_domain", "branch_id",   # admin / marketing fields
                   "campaign_code", "marketing_segment", "document_preference")
# ---------------------------------------------------------------------------

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(lubridate)
  library(stringr)
  library(tidyr)
})
stopifnot(packageVersion("dplyr") >= "1.1.0")   # needed for join_by()

# Find the raw data folder: the one folder that contains customers.parquet
if (is.null(data_dir)) {
  hits <- unique(dirname(list.files(c("/kaggle/input", "data"), pattern = "^customers\\.parquet$",
                                    recursive = TRUE, full.names = TRUE)))
  if (length(hits) != 1) stop("Found ", length(hits), " folders with customers.parquet: ",
                              paste(hits, collapse = ", "), ". Set data_dir in Cell 1.")
  data_dir <- hits
}
out_dir <- if (dir.exists("/kaggle/working")) file.path("/kaggle/working", run_folder) else run_folder
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

cat("Raw data:", data_dir, "\nOutput:  ", out_dir, "\n")

Raw data: /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public 
Output:   /kaggle/working/prep 


In [36]:
# %% Cell 2: Helper functions =================================================

id_cols   <- c("row_id", "policy_id", "customer_id", "address_id", "claim_id",
               "repairer_id", "third_party_id")
date_cols <- c("period_start", "period_end", "loss_date", "report_date", "date_of_birth")

# Read one raw file from data_dir: IDs as text, dates as Date,
# other text lowercased and trimmed (IDs are never changed).
f <- function(name) {
  path <- file.path(data_dir, paste0(name, ".parquet"))
  if (!file.exists(path)) stop("Missing file: ", path)
  df <- as_tibble(read_parquet(path)) |>
    mutate(across(any_of(id_cols), as.character),
           across(any_of(date_cols), as.Date))
  text_cols <- setdiff(names(df)[sapply(df, is.character)], id_cols)
  df |> mutate(across(all_of(text_cols), \(x) str_trim(str_to_lower(x))))
}

# Restore leading zeros lost when a file stored IDs as numbers.
# Pads only all-digit IDs shorter than the reference IDs, and only when the
# reference IDs are all digits of a single width - otherwise does nothing.
repair_ids <- function(x, ref, label) {
  ref <- unique(ref[!is.na(ref)])
  w <- unique(nchar(ref))
  if (length(w) != 1 || !all(grepl("^[0-9]+$", ref))) return(x)
  short <- !is.na(x) & grepl("^[0-9]+$", x) & nchar(x) < w
  if (any(short)) cat("  Repaired", sum(short), label, "values missing leading zeros\n")
  x[short] <- paste0(strrep("0", w - nchar(x[short])), x[short])
  x
}

# Stop if a lookup table has duplicate keys (a join would multiply rows)
check_unique <- function(df, keys, label) {
  dups <- df |> count(across(all_of(keys))) |> filter(n > 1)
  if (nrow(dups) > 0) { print(head(dups)); stop(label, " has duplicated keys on ", paste(keys, collapse = " + ")) }
  cat("  OK:", label, "is unique on", paste(keys, collapse = " + "), "\n")
}

# Last covered day of a calendar year under the data's convention
year_last_day <- function(y) {
  if (leap_ends_dec30) if_else(leap_year(y), make_date(y, 12, 30), make_date(y, 12, 31))
  else make_date(y, 12, 31)
}

# Policy rows -> exposure in years. Works for any years:
#   1. A row crossing a year end is split into one piece per calendar year.
#      Pieces get row_id "<row_id>_<year>"; orig_row_id keeps the original
#      (predictions are summed back to orig_row_id for submission).
#   2. Each piece is cut at the year's last covered day (30 Dec in leap years).
#   3. Days already covered by an earlier piece of the same policy are removed.
#   4. Exposure = days / days_per_year, then hard checks that nothing exceeds 1.
clean_policies <- function(df, name) {
  if (any(df$period_end < df$period_start)) stop(name, ": rows end before they start")

  n_in <- nrow(df)
  df <- df |>
    mutate(orig_row_id     = row_id,
           exposure_kaggle = exposure,
           .y0 = year(period_start), .y1 = year(period_end),
           .n_years = .y1 - .y0 + 1) |>
    uncount(.n_years, .id = ".k") |>
    mutate(.y          = .y0 + .k - 1,
           .split      = .y1 > .y0,
           period_start = if_else(.k == 1, period_start, make_date(.y, 1, 1)),
           period_end   = if_else(.y == .y1, period_end, make_date(.y, 12, 31)),
           .trimmed     = period_end > year_last_day(.y),
           period_end   = pmin(period_end, year_last_day(.y)),
           row_id          = if_else(.split, paste0(orig_row_id, "_", .y), orig_row_id),
           exposure_kaggle = if_else(.split, NA_real_, exposure_kaggle))

  n_split   <- n_distinct(df$orig_row_id[df$.split])
  n_trimmed <- sum(df$.trimmed)

  df <- df |>
    mutate(.row_order = row_number()) |>
    arrange(policy_id, period_start, period_end) |>
    group_by(policy_id) |>
    mutate(prev_cover_end = lag(cummax(as.numeric(period_end))),
           eff_start      = pmax(as.numeric(period_start), prev_cover_end + 1, na.rm = TRUE),
           overlap_days   = pmax(0, eff_start - as.numeric(period_start))) |>
    ungroup() |>
    mutate(days     = pmax(0, as.numeric(period_end) - eff_start + 1),
           exposure = days / days_per_year) |>
    arrange(.row_order)

  # ---- Hard checks: exposure can never exceed 1 ---------------------------
  if (any(df$exposure > 1)) stop(name, ": a row exceeds 1 year - check the settings in Cell 1")
  per_year <- df |> group_by(policy_id, yr = year(period_start)) |>
    summarise(t = sum(exposure), d = sum(days), .groups = "drop")
  if (any(per_year$t > 1 + 1e-9)) stop(name, ": a policy exceeds 1 year of exposure in a calendar year")

  # ---- Convention check: full policy-years should cover days_per_year days --
  near_full <- per_year$d[per_year$d >= days_per_year - 5]
  typical   <- if (length(near_full)) as.numeric(names(which.max(table(near_full)))) else NA
  if (!is.na(typical) && typical != days_per_year) {
    warning(name, ": full policy-years usually cover ", typical, " days, not ", days_per_year,
            " - the date convention may differ; review the settings in Cell 1")
  }

  cat(sprintf("  %-16s rows in: %9s | split across years: %5s | cut at leap-year 30 Dec: %5s | overlap days removed on: %5s | zero exposure: %4s | full year = %s days | max: %.4f\n",
              name, format(n_in, big.mark = ","), n_split, n_trimmed, sum(df$overlap_days > 0),
              sum(df$exposure == 0), typical, max(df$exposure)))
  n_diff <- sum(abs(df$exposure - df$exposure_kaggle) > 0.01, na.rm = TRUE)
  if (n_diff > 0) cat("    exposure differs from Kaggle's by >0.01 on", n_diff, "unsplit rows\n")
  if (any(df$exposure == 0)) warning(name, ": ", sum(df$exposure == 0), " rows have zero exposure - inspect before modelling")

  df |> select(-starts_with("."), -c(prev_cover_end, eff_start, overlap_days, days), -any_of(drop_cols))
}

In [37]:
# %% Cell 3: Load reference tables and claims =================================

customers  <- f("customers")
addresses  <- f("addresses")
weather    <- f("weather")
claims     <- f("claims")
sample_sub <- f("sample_submission")

weather$address_id <- repair_ids(weather$address_id, addresses$address_id, "weather address_id")

cat("Rows - customers:", nrow(customers), "| addresses:", nrow(addresses),
    "| weather:", nrow(weather), "| claims:", nrow(claims), "| sample_submission:", nrow(sample_sub), "\n")

Rows - customers: 545240 | addresses: 575983 | weather: 2879915 | claims: 163150 | sample_submission: 626639 


In [38]:
# %% Cell 4: Load and clean policy files (always from the raw files) ==========

load_policies <- function(file, name) {
  df <- f(file)
  df$customer_id <- repair_ids(df$customer_id, customers$customer_id, paste(name, "customer_id"))
  df$address_id  <- repair_ids(df$address_id,  addresses$address_id,  paste(name, "address_id"))
  clean_policies(df, name)
}

train <- setNames(lapply(lines, \(l) load_policies(paste0("policies_", l), paste("train", l))), lines)
test  <- setNames(lapply(lines, \(l) load_policies(paste0("test_", l),     paste("test", l))),  lines)

all_policy_ids   <- unlist(lapply(c(train, test), \(d) d$policy_id))
claims$policy_id <- repair_ids(claims$policy_id, all_policy_ids, "claims policy_id")

  Repaired 98110 train motor customer_id values missing leading zeros
  train motor      rows in:   979,968 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 4474 unsplit rows
  train buildings  rows in:   707,726 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 3365 unsplit rows
  train contents   rows in:   721,617 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 3434 unsplit rows
  Repaired 25554 test motor customer_id values missing leading zeros
  test motor       rows in:   254,986 | split across years:     0 | cut at leap-year 30 Dec: 

In [39]:
# %% Cell 5: Key checks ========================================================

check_unique(customers, "customer_id", "customers")
check_unique(addresses, "address_id", "addresses")
check_unique(weather, c("address_id", "year"), "weather")
check_unique(claims, "claim_id", "claims")
for (l in lines) {
  check_unique(train[[l]], "row_id", paste("train", l))
  check_unique(test[[l]],  "row_id", paste("test", l))
}

odd_lines <- setdiff(unique(claims$line), lines)
if (length(odd_lines)) warning("Unexpected claim line values: ", paste(odd_lines, collapse = ", "))

shared_pids <- bind_rows(lapply(lines, \(l) distinct(train[[l]], policy_id) |> mutate(line = l))) |>
  count(policy_id) |> filter(n > 1)
cat("  policy_ids in more than one line:", nrow(shared_pids), "\n")

test_ids <- unique(unlist(lapply(test, \(d) d$orig_row_id)))   # original rows, before any year split
if (!setequal(test_ids, sample_sub$row_id) || length(test_ids) != nrow(sample_sub)) {
  stop("Test row_ids don't match sample_submission (", length(test_ids), " vs ", nrow(sample_sub), ")")
}
cat("  OK: all", format(length(test_ids), big.mark = ","), "test row_ids match sample_submission\n")

  OK: customers is unique on customer_id 
  OK: addresses is unique on address_id 
  OK: weather is unique on address_id + year 
  OK: claims is unique on claim_id 
  OK: train motor is unique on row_id 
  OK: test motor is unique on row_id 
  OK: train buildings is unique on row_id 
  OK: test buildings is unique on row_id 
  OK: train contents is unique on row_id 
  OK: test contents is unique on row_id 
  policy_ids in more than one line: 0 
  OK: all 626,639 test row_ids match sample_submission


In [40]:
# %% Cell 6: Match claims to policy rows ======================================
# A claim belongs to the row of its policy whose period contains its loss_date.
# If two rows qualify (shared changeover day), the earlier row is used - the
# same row that keeps that day's exposure in clean_policies(), so claims and
# exposure stay consistent.
# Claims outside every row are set aside, with the reason.

match_claims <- function(line_name) {
  cl  <- claims |> filter(line == line_name)
  pol <- train[[line_name]]

  m <- cl |> inner_join(pol |> select(row_id, orig_row_id, policy_id, period_start, period_end),
                        by = join_by(policy_id, between(loss_date, period_start, period_end)))
  doubles  <- m |> add_count(claim_id, name = "n_rows") |> filter(n_rows > 1)
  resolved <- m |> group_by(claim_id) |> slice_min(period_start, n = 1, with_ties = FALSE) |> ungroup()

  ranges <- pol |> group_by(policy_id) |>
    summarise(first_start = min(period_start), last_end = max(period_end), .groups = "drop")
  unmatched <- cl |> anti_join(resolved, by = "claim_id") |> left_join(ranges, by = "policy_id") |>
    mutate(reason = case_when(is.na(first_start)       ~ "policy_id not in policy file",
                              loss_date < first_start  ~ "before first row",
                              loss_date > last_end     ~ "after last row",
                              TRUE                     ~ "in a gap between rows")) |>
    select(-first_start, -last_end)

  cat(sprintf("  %-10s claims: %8s | matched: %8s | double-matched: %5s | unmatched: %5s\n", line_name,
              format(nrow(cl), big.mark = ","), format(nrow(resolved), big.mark = ","),
              n_distinct(doubles$claim_id), nrow(unmatched)))
  if (nrow(unmatched)) print(count(unmatched, reason, status))

  list(matched = resolved, doubles = doubles, unmatched = unmatched)
}

claim_results    <- setNames(lapply(lines, match_claims), lines)
claims_matched   <- bind_rows(lapply(claim_results, `[[`, "matched"))
claims_doubles   <- bind_rows(lapply(claim_results, `[[`, "doubles"))
claims_unmatched <- bind_rows(lapply(claim_results, `[[`, "unmatched"))
stopifnot(!anyDuplicated(claims_matched$claim_id))

  motor      claims:  112,606 | matched:  112,309 | double-matched:     0 | unmatched:   297
# A tibble: 3 × 3
  reason           status         n
  <chr>            <chr>      <int>
1 before first row closed       283
2 before first row open          11
3 before first row repudiated     3
  buildings  claims:   26,442 | matched:   26,375 | double-matched:     0 | unmatched:    67
# A tibble: 2 × 3
  reason           status     n
  <chr>            <chr>  <int>
1 before first row closed    64
2 before first row open       3
  contents   claims:   24,102 | matched:   24,052 | double-matched:     0 | unmatched:    50
# A tibble: 3 × 3
  reason           status         n
  <chr>            <chr>      <int>
1 before first row closed        48
2 before first row open           1
3 before first row repudiated     1


In [41]:
# %% Cell 7: Join customers, addresses and weather ============================
# Weather joins on address and the calendar year of period_start. If that year
# is missing, the address's latest earlier year is used (weather_imputed = TRUE).

weather_cols <- setdiff(names(weather), c("address_id", "year"))
weather_fb   <- weather |> rename(fb_year = year) |> rename_with(\(x) paste0(x, "_fb"), all_of(weather_cols))

add_weather <- function(pol) {
  pol <- pol |> mutate(weather_year = year(period_start))
  fb  <- pol |> select(row_id, address_id, weather_year) |>
    inner_join(weather_fb, by = join_by(address_id, closest(weather_year >= fb_year))) |>
    select(row_id, ends_with("_fb"))
  pol |>
    left_join(weather, by = join_by(address_id, weather_year == year)) |>
    mutate(weather_imputed = is.na(.data[[weather_cols[1]]])) |>
    left_join(fb, by = "row_id") |>
    mutate(across(all_of(weather_cols), \(x) coalesce(x, get(paste0(cur_column(), "_fb"))))) |>
    select(-ends_with("_fb"))
}

build_features <- function(pol, name) {
  out <- pol |>
    left_join(customers |> mutate(.cust = TRUE), by = "customer_id") |>
    left_join(addresses |> mutate(.addr = TRUE), by = "address_id") |>
    add_weather()

  cat(sprintf("  %-16s no customer: %6s | no address: %6s | weather filled from earlier year: %6s | weather still missing: %6s\n",
              name, sum(is.na(out$.cust)), sum(is.na(out$.addr)), sum(out$weather_imputed),
              sum(is.na(out[[weather_cols[1]]]))))
  out <- out |> select(-.cust, -.addr, -any_of(drop_cols))
  stopifnot(nrow(out) == nrow(pol), !anyDuplicated(out$row_id))   # joins never add or drop rows
  out
}

train_full <- setNames(lapply(lines, \(l) build_features(train[[l]], paste("train", l))), lines)
test_full  <- setNames(lapply(lines, \(l) build_features(test[[l]],  paste("test", l))),  lines)

for (l in lines) {
  diff_cols <- union(setdiff(names(train_full[[l]]), names(test_full[[l]])),
                     setdiff(names(test_full[[l]]), names(train_full[[l]])))
  if (length(diff_cols)) warning(l, ": train/test column mismatch: ", paste(diff_cols, collapse = ", "))
  else cat("  OK:", l, "train and test have the same columns\n")
}

  train motor      no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  train buildings  no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  train contents   no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  test motor       no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  test buildings   no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  test contents    no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  OK: motor train and test have the same columns
  OK: buildings train and test have the same columns
  OK: contents train and test have the same columns


In [42]:
# %% Cell 8: Save outputs, summary and session info ===========================

for (l in lines) {
  write_parquet(train_full[[l]], file.path(out_dir, paste0("train_", l, ".parquet")))
  write_parquet(test_full[[l]],  file.path(out_dir, paste0("test_",  l, ".parquet")))
}
write_parquet(claims_matched,   file.path(out_dir, "claims_matched.parquet"))
write_parquet(claims_doubles,   file.path(out_dir, "claims_double_matched.parquet"))
write_parquet(claims_unmatched, file.path(out_dir, "claims_unmatched.parquet"))

summary_tbl <- bind_rows(lapply(lines, \(l) tibble(
  line              = l,
  train_rows        = n_distinct(train_full[[l]]$orig_row_id),
  train_pieces      = nrow(train_full[[l]]),
  test_rows         = n_distinct(test_full[[l]]$orig_row_id),
  test_pieces       = nrow(test_full[[l]]),
  train_exposure    = sum(train_full[[l]]$exposure),
  claims            = sum(claims$line == l),
  claims_matched    = nrow(claim_results[[l]]$matched),
  claims_double     = n_distinct(claim_results[[l]]$doubles$claim_id),
  claims_unmatched  = nrow(claim_results[[l]]$unmatched)
)))
write.csv(summary_tbl, file.path(out_dir, "phase1_summary.csv"), row.names = FALSE)
writeLines(c(paste("Run:", format(Sys.time(), "%Y-%m-%d %H:%M %Z")), paste("Data:", data_dir),
             capture.output(sessionInfo())), file.path(out_dir, "session_info.txt"))

print(as.data.frame(summary_tbl))
cat("\nPhase 1 complete. Files in", out_dir, ":\n"); print(list.files(out_dir))

       line train_rows train_pieces test_rows test_pieces train_exposure claims claims_matched claims_double claims_unmatched
1     motor     979968       979968    254986      254986       787157.5 112606         112309             0              297
2 buildings     707726       707726    184155      184155       590560.2  26442          26375             0               67
3  contents     721617       721617    187498      187498       590859.3  24102          24052             0               50

Phase 1 complete. Files in /kaggle/working/prep :
 [1] "claims_double_matched.parquet" "claims_matched.parquet"        "claims_unmatched.parquet"      "phase1_summary.csv"            "session_info.txt"              "test_buildings.parquet"       
 [7] "test_contents.parquet"         "test_motor.parquet"            "train_buildings.parquet"       "train_contents.parquet"        "train_motor.parquet"          


In [43]:
train$motor %>%
  filter(abs(exposure - exposure_kaggle) > 0.01) %>%
  select(policy_id, period_start, period_end, exposure, exposure_kaggle) %>%
  head(15)

policy_id,period_start,period_end,exposure,exposure_kaggle
<chr>,<date>,<date>,<dbl>,<dbl>
M58879003,2021-01-01,2021-12-31,1.0000000,1.0139
M15933607,2021-01-01,2021-12-31,1.0000000,1.0139
M88789901,2021-02-08,2021-12-31,0.8958904,327.0000
M26530985,2021-01-01,2021-12-31,1.0000000,1.0139
M60758696,2021-01-01,2021-12-31,1.0000000,1.0139
M59020701,2021-01-01,2021-12-31,1.0000000,1.0139
M79277737,2021-01-01,2021-12-31,1.0000000,1.0139
M37772034,2021-01-01,2021-12-31,1.0000000,1.0139
M73329653,2021-01-01,2021-12-31,1.0000000,365.0000


In [44]:
bind_rows(lapply(lines, \(l) train[[l]] %>% mutate(line = l))) %>%
  filter(!is.na(exposure_kaggle), abs(exposure - exposure_kaggle) > 0.01) %>%
  mutate(days = round(exposure * days_per_year),
         error_type = case_when(
           abs(exposure_kaggle - days / 360) < 0.001 ~ "360-day basis",
           abs(exposure_kaggle - days) < 0.5         ~ "recorded in days",
           TRUE                                      ~ "other")) %>%
  count(line, year = year(period_start), error_type)

line,year,error_type,n
<chr>,<dbl>,<chr>,<int>
buildings,2021,360-day basis,692
buildings,2021,recorded in days,156
buildings,2022,360-day basis,662
buildings,2022,recorded in days,164
buildings,2023,360-day basis,682
buildings,2023,recorded in days,173
buildings,2024,360-day basis,673
buildings,2024,recorded in days,163
contents,2021,360-day basis,662


Phase 2

In [45]:
# =============================================================================
# Phase 2a: Check every column against the dataset description
#
# Reads the ORIGINAL files (so columns dropped in Phase 1, like renewal_decision
# and document_preference, and the claims file are checked too) and tests each
# column against what the description says it should contain:
#
#   A. Categories   - values outside the documented list (e.g. "t", "w"),
#                     with a text-based suggestion; case/spacing variants
#   B. Codes/sets   - voluntary_excess, flood_zone, vehicle_value_group, region
#   C. Ranges       - indices (0-1, 0-10), telematics 1-100, counts >= 0,
#                     whole numbers where counts are expected, ages, years
#   D. Yes/no flags - stored as TRUE/FALSE, not mixed codes
#   E. Cross-checks - telematics vs opt-in, licence vs age, NCB vs licence,
#                     claims: peril valid for line, status, dates, amounts
#   F. Free text    - spelling variants in undocumented text columns
#                     (occupation, make_model, vehicle_colour, ...)
#   G. Missing values, and test categories never seen in training
#
# Read-only. No hard-coded years: the prediction year is read from the test
# files. 
# =============================================================================

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(tidyr)
})

# ---- Settings: the only lines to change for a different dataset ----------
raw_dir <- NULL        # raw data folder; NULL = the one folder holding customers.parquet
lines   <- c("motor", "buildings", "contents")
# ---------------------------------------------------------------------------

if (is.null(raw_dir)) {
  hits <- unique(dirname(list.files(c("/kaggle/input", "data"), pattern = "^customers\\.parquet$",
                                    recursive = TRUE, full.names = TRUE)))
  if (length(hits) != 1) stop("Found ", length(hits), " data folders - set raw_dir")
  raw_dir <- hits
}

norm <- function(x) gsub("[ -]+", "_", tolower(trimws(as.character(x))))   # "Non-Standard " -> "non_standard"

read_raw <- function(name) {
  df <- as_tibble(read_parquet(file.path(raw_dir, paste0(name, ".parquet"))))
  df |> mutate(across(any_of(c("period_start", "period_end", "loss_date", "report_date", "date_of_birth")), as.Date))
}

file_names <- c("customers", "addresses", "weather", "claims",
           paste0("policies_", lines), paste0("test_", lines))
raw <- setNames(lapply(file_names, read_raw), file_names)
cat("Read:", paste(file_names, collapse = ", "), "\n")

max_year <- max(sapply(paste0("test_", lines), \(f) max(as.numeric(format(raw[[f]]$period_start, "%Y")), na.rm = TRUE)))
cat("Prediction year detected from test files:", max_year, "\n")

# =============================================================================
# Specification, taken from the dataset description
# =============================================================================
categories <- list(
  marital_status      = c("single", "married", "divorced", "widowed"),
  credit_band         = c("a", "b", "c", "d", "e"),
  document_preference = c("email", "post", "app"),
  region              = sprintf("r%02d", 1:12),
  urban_density       = c("rural", "suburban", "urban", "dense_urban"),
  soil_type           = c("clay", "sand", "loam", "chalk", "rock"),
  property_type       = c("detached", "semi_detached", "terraced", "flat"),
  roof_type           = c("pitched", "flat"),
  energy_rating       = c("a", "b", "c", "d", "e", "f", "g"),
  channel             = c("direct", "comparison_site", "broker"),
  payment_method      = c("annual", "monthly"),
  renewal_decision    = c("renewed", "lapsed_by_customer", "cancelled", "declined_by_insurer", "mid_term_change"),
  body_type           = c("hatchback", "saloon", "estate", "suv", "coupe"),
  transmission        = c("manual", "automatic"),
  fuel_type           = c("petrol", "diesel", "hybrid", "electric"),
  cover_type          = c("comprehensive", "third_party_fire_theft", "third_party_only"),
  overnight_parking   = c("garage", "driveway", "street"),
  vehicle_use         = c("social", "commuting", "business"),
  construction_type   = c("brick", "stone", "timber", "non_standard"),
  occupancy           = c("owner_occupied", "let"),
  tenure              = c("owner", "renter"),
  security_level      = c("standard", "locks", "alarm_and_locks"),
  line                = lines,
  status              = c("closed", "open", "repudiated")
)
perils <- list(motor     = c("own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"),
               buildings = c("storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"),
               contents  = c("theft", "escape_of_water", "accidental_damage", "flood"))

value_sets <- list(voluntary_excess = c(0, 250, 500, 1000), flood_zone = 1:5, vehicle_value_group = 1:20)

ranges <- list(   # inclusive min, max
  telematics_score = c(1, 100), wind_exposure_index = c(0, 10), dry_summer_index = c(0, 1),
  wildfire_index = c(0, 1), hail_days = c(0, 366), freezing_days = c(0, 366), days_unoccupied = c(0, 366),
  missed_payments_12m = c(0, 12), driver_age = c(17, 100), policyholder_age = c(18, 110),
  year_built = c(1000, max_year), customer_since_year = c(1900, max_year))

non_negative <- c("prior_claims_5y", "licence_points", "years_at_address", "bedrooms", "renewal_count",
                  "customer_service_contacts", "rainfall_mm", "ncb_years", "vehicle_age", "annual_mileage",
                  "high_value_items", "days_quote_to_inception", "compulsory_excess", "distance_to_river_km",
                  "engine_size_cc", "years_licensed", "discount_applied", "paid_to_date", "incurred_to_date",
                  "recoveries")
positive     <- c("household_size", "written_premium", "sum_insured", "seats", "exposure")
whole_number <- c("household_size", "prior_claims_5y", "licence_points", "bedrooms", "missed_payments_12m",
                  "renewal_count", "customer_service_contacts", "ncb_years", "seats", "high_value_items",
                  "hail_days", "freezing_days", "flood_zone", "vehicle_value_group", "days_quote_to_inception",
                  "year_built", "customer_since_year", "voluntary_excess")
yes_no <- c("marketing_consent", "basement", "trees_within_5m", "listed_building", "solar_panels",
            "new_business", "driver_assist", "named_young_driver", "telematics_opt_in", "smoke_alarms",
            "accidental_damage_cover", "mortgage", "works_from_home", "lodgers", "bicycle_cover")

# Columns the description leaves open: checked for spelling variants only
free_text <- c("occupation", "email_domain", "previous_insurer", "campaign_code", "branch_id",
               "marketing_segment", "make_model", "vehicle_colour")

# =============================================================================
# Helpers
# =============================================================================
issues <- list()
add_issue <- function(file, column, check, bad, values = NULL) {
  n_bad <- sum(bad, na.rm = TRUE)
  if (n_bad == 0) return(invisible())
  ex <- if (!is.null(values)) {
    tb <- sort(table(as.character(values[which(bad)]), useNA = "ifany"), decreasing = TRUE)
    paste(head(paste0(names(tb), " (", tb, ")"), 6), collapse = ", ")
  } else ""
  issues[[length(issues) + 1]] <<- tibble(file = file, column = column, check = check,
                                          n_bad = n_bad, pct = round(100 * n_bad / length(bad), 3),
                                          examples = ex)
}

suggest <- function(v, docs) {
  if (is.na(v) || v == "na") return("(blank)")
  initials <- sapply(strsplit(docs, "_"), \(w) paste(substr(w, 1, 1), collapse = ""))
  out <- unique(c(docs[startsWith(docs, v)], docs[initials == v],
                  if (nchar(v) >= 4) docs[adist(v, docs)[1, ] <= 2] else character(0)))
  if (length(out)) paste(out, collapse = "/") else "?"
}

as_flag <- function(x) if (is.logical(x)) x else norm(x) %in% c("true", "t", "yes", "y", "1")

# =============================================================================
# A-D. Column-by-column checks
# =============================================================================
category_detail <- list()

for (f in file_names) {
  df <- raw[[f]]
  for (col in names(df)) {
    x <- df[[col]]

    # A. Categories
    if (col %in% names(categories)) {
      docs <- categories[[col]]
      xn   <- norm(x)
      bad  <- is.na(x) | !xn %in% docs
      add_issue(f, col, "value not in description", bad, xn)
      # the same category written more than one way, e.g. "Brick" and "brick "
      multi <- tibble(r = as.character(x), v = xn) |> distinct() |> count(v, name = "forms") |> filter(forms > 1) |> pull(v)
      if (length(multi)) add_issue(f, col, "same value written in different case/spacing",
                                   xn %in% multi, as.character(x))
      if (any(bad)) {
        category_detail[[paste(f, col)]] <- tibble(file = f, column = col, value = xn[bad]) |>
          count(file, column, value, name = "rows") |>
          mutate(suggestion = sapply(value, suggest, docs = docs))
      }
    }

    # B. Fixed sets of allowed values
    if (col %in% names(value_sets)) add_issue(f, col, paste("not one of", paste(value_sets[[col]], collapse = "/")),
                                              !is.na(x) & !x %in% value_sets[[col]], x)

    # C. Ranges, signs and whole numbers (numeric columns only)
    if (is.numeric(x)) {
      if (col %in% names(ranges)) add_issue(f, col, sprintf("outside %s to %s", ranges[[col]][1], ranges[[col]][2]),
                                            !is.na(x) & (x < ranges[[col]][1] | x > ranges[[col]][2]), x)
      if (col %in% non_negative) add_issue(f, col, "negative", !is.na(x) & x < 0, x)
      if (col %in% positive)     add_issue(f, col, "zero or negative", !is.na(x) & x <= 0, x)
      if (col %in% whole_number) add_issue(f, col, "not a whole number", !is.na(x) & abs(x - round(x)) > 1e-9, x)
    } else if (col %in% c(names(ranges), non_negative, positive, whole_number, names(value_sets))) {
      add_issue(f, col, paste("stored as", class(x)[1], "not a number"), rep(TRUE, length(x)), x)
    }

    # D. Yes/no flags
    if (col %in% yes_no && !is.logical(x)) {
      add_issue(f, col, paste("yes/no flag stored as", class(x)[1]), rep(TRUE, length(x)), x)
    }
  }

  # Date of birth -> age at the prediction year
  if ("date_of_birth" %in% names(df)) {
    age <- max_year - as.numeric(format(df$date_of_birth, "%Y"))
    add_issue(f, "date_of_birth", sprintf("age in %d outside 16-110", max_year), !is.na(age) & (age < 16 | age > 110), age)
  }
}

# =============================================================================
# E. Cross-checks between columns
# =============================================================================
for (l in lines) for (f in c(paste0("policies_", l), paste0("test_", l))) {
  df <- raw[[f]]
  if (all(c("telematics_score", "telematics_opt_in") %in% names(df))) {
    opt <- as_flag(df$telematics_opt_in)
    add_issue(f, "telematics_score", "score present but not opted in", !is.na(df$telematics_score) & !opt)
    add_issue(f, "telematics_score", "opted in but score blank", is.na(df$telematics_score) & opt)
  }
  if (all(c("years_licensed", "driver_age") %in% names(df)))
    add_issue(f, "years_licensed", "more than driver_age - 16", df$years_licensed > df$driver_age - 16, df$years_licensed)
  if (all(c("ncb_years", "years_licensed") %in% names(df)))
    add_issue(f, "ncb_years", "more than years_licensed", df$ncb_years > df$years_licensed, df$ncb_years)
  if (all(c("engine_size_cc", "fuel_type") %in% names(df)))
    add_issue(f, "engine_size_cc", "zero for a non-electric vehicle", df$engine_size_cc <= 0 & norm(df$fuel_type) != "electric")
  if ("period_start" %in% names(df))
    add_issue(f, "period_end", "before period_start", df$period_end < df$period_start)
}

cl <- raw$claims
cl_line <- norm(cl$line); cl_peril <- norm(cl$peril); cl_status <- norm(cl$status)
bad_peril <- !mapply(\(p, l) l %in% names(perils) && p %in% perils[[l]], cl_peril, cl_line)
add_issue("claims", "peril", "peril not listed for its line", bad_peril, paste(cl_line, cl_peril, sep = ": "))
add_issue("claims", "report_date", "before loss_date", cl$report_date < cl$loss_date)
add_issue("claims", "report_date", "after the latest report date in the file (valuation)", cl$report_date > max(cl$report_date, na.rm = TRUE))
add_issue("claims", "incurred_to_date", "less than paid_to_date", cl$incurred_to_date < cl$paid_to_date - 0.01)
add_issue("claims", "incurred_to_date", "closed claim with incurred != paid", cl_status == "closed" & abs(cl$incurred_to_date - cl$paid_to_date) > 0.01)
add_issue("claims", "paid_to_date", "declined claim with a payment", cl_status == "repudiated" & cl$paid_to_date > 0)
add_issue("claims", "recoveries", "recoveries larger than paid", cl$recoveries > cl$paid_to_date + 0.01)

# =============================================================================
# F. Spelling variants in free-text columns
# =============================================================================
variant_list <- list()
for (f in file_names) for (col in intersect(free_text, names(raw[[f]]))) {
  vals <- raw[[f]] |> count(value = as.character(.data[[col]]), name = "rows") |> filter(!is.na(value))
  vals <- vals |> mutate(key = gsub("[^a-z0-9]", "", tolower(value)))
  # same letters once case, spaces and punctuation are ignored
  grp <- vals |> group_by(key) |> filter(n() > 1) |> ungroup()
  # one-letter typos between values of 5+ characters (only for manageable numbers of values)
  typo <- tibble()
  if (nrow(vals) <= 400) {
    v <- vals$value[nchar(vals$key) >= 5]
    if (length(v) > 1) {
      d <- adist(tolower(v)); d[upper.tri(d, diag = TRUE)] <- NA
      idx <- which(d == 1, arr.ind = TRUE)
      if (nrow(idx)) typo <- tibble(a = v[idx[, 1]], b = v[idx[, 2]])
    }
  }
  if (nrow(grp) || nrow(typo)) {
    variant_list[[paste(f, col)]] <- tibble(
      file = f, column = col, distinct_values = nrow(vals),
      same_once_normalised = if (nrow(grp)) paste(head(paste0(grp$value, " (", grp$rows, ")"), 8), collapse = ", ") else "",
      one_letter_apart = if (nrow(typo)) paste(head(paste(typo$a, "~", typo$b), 6), collapse = ", ") else "")
  }
}

# =============================================================================
# G. Missing values, and test categories unseen in training
# =============================================================================
na_tbl <- bind_rows(lapply(file_names, \(f) {
  n <- colSums(is.na(raw[[f]]))
  tibble(file = f, column = names(n), missing = n, pct = round(100 * n / nrow(raw[[f]]), 2)) |> filter(missing > 0)
}))

unseen_tbl <- bind_rows(lapply(lines, \(l) {
  tr <- raw[[paste0("policies_", l)]]; te <- raw[[paste0("test_", l)]]
  cols <- intersect(names(te)[sapply(te, \(x) is.character(x) || is.factor(x))], names(tr))
  cols <- cols[!grepl("_id$", cols)]
  bind_rows(lapply(cols, \(c) {
    new <- setdiff(unique(norm(te[[c]])), unique(norm(tr[[c]])))
    tibble(line = l, column = c, new_values = length(new), test_rows = sum(norm(te[[c]]) %in% new),
           examples = paste(head(new, 5), collapse = ", "))
  })) |> filter(new_values > 0)
}))

# =============================================================================
# Report
# =============================================================================
options(width = 250)
cat("\n==================== SUMMARY: every failed check ====================\n")
if (length(issues)) print(bind_rows(issues) |> arrange(file, column), n = Inf, width = Inf) else cat("No issues found\n")

cat("\n==================== A. Undocumented category values, with suggestions ====================\n")
if (length(category_detail)) print(bind_rows(category_detail), n = Inf) else cat("None\n")

cat("\n==================== F. Possible spelling variants in free-text columns ====================\n")
if (length(variant_list)) print(bind_rows(variant_list), n = Inf, width = Inf) else cat("None\n")

cat("\n==================== G1. Missing values ====================\n")
if (nrow(na_tbl)) print(na_tbl, n = Inf) else cat("None\n")

cat("\n==================== G2. Test values never seen in training ====================\n")
if (nrow(unseen_tbl)) print(unseen_tbl, n = Inf, width = Inf) else cat("None\n")

Read: customers, addresses, weather, claims, policies_motor, policies_buildings, policies_contents, test_motor, test_buildings, test_contents 
Prediction year detected from test files: 2025 

==================== SUMMARY: every failed check ====================
# A tibble: 20 × 6
   file               column            check                                         n_bad     pct examples                                                                                  
   <chr>              <chr>             <chr>                                         <int>   <dbl> <chr>                                                                                     
 1 addresses          energy_rating     value not in description                      14443   2.51  NA (14443)                                                                                
 2 addresses          roof_type         value not in description                      14401   2.5   NA (14401)                                    

In [46]:
# =============================================================================
# Phase 2a (part 2): Undocumented category check - every categorical column
#
# For every column with values not in the data description, shows:
#   - each value, documented or not, with train and test row counts
#   - a text-based suggestion for undocumented codes (abbreviation, initials
#     or a likely typo of a documented value, e.g. "t" -> "timber")
#   - a claims profile for each value (exposure, claim frequency, average
#     cost per claim, plus any extra profile columns available) so an
#     undocumented code can be compared with the documented values it might be
#   - the raw values before lowercasing, from the original files
#
# Read-only: changes nothing. Works for any years or dataset with the same
# columns;
# =============================================================================

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(tidyr)
})

# ---- Settings: the only lines to change for a different dataset ----------
run_folder <- "prep"     # folder Phase 1 saved to, e.g. "prep_week2"
raw_dir    <- NULL       # raw data folder; NULL = the one folder holding customers.parquet
lines      <- c("motor", "buildings", "contents")
# ---------------------------------------------------------------------------

prep_dir <- if (dir.exists("/kaggle/working")) file.path("/kaggle/working", run_folder) else run_folder
if (is.null(raw_dir)) {
  hits <- unique(dirname(list.files(c("/kaggle/input", "data"), pattern = "^customers\\.parquet$",
                                    recursive = TRUE, full.names = TRUE)))
  raw_dir <- if (length(hits) == 1) hits else NA
}

load_prep <- function(name) as_tibble(read_parquet(file.path(prep_dir, paste0(name, ".parquet"))))
train  <- setNames(lapply(lines, \(l) load_prep(paste0("train_", l))), lines)
test   <- setNames(lapply(lines, \(l) load_prep(paste0("test_",  l))), lines)
claims <- load_prep("claims_matched")
options(width = 250)   # print each profile table on one line

# ---- Documented values (from the data description) -------------------------
documented <- list(
  marital_status    = c("single", "married", "divorced", "widowed"),
  credit_band       = c("a", "b", "c", "d", "e"),
  region            = sprintf("r%02d", 1:12),
  urban_density     = c("rural", "suburban", "urban", "dense_urban"),
  soil_type         = c("clay", "sand", "loam", "chalk", "rock"),
  property_type     = c("detached", "semi_detached", "terraced", "flat"),
  roof_type         = c("pitched", "flat"),
  energy_rating     = c("a", "b", "c", "d", "e", "f", "g"),
  channel           = c("direct", "comparison_site", "broker"),
  payment_method    = c("annual", "monthly"),
  body_type         = c("hatchback", "saloon", "estate", "suv", "coupe"),
  transmission      = c("manual", "automatic"),
  fuel_type         = c("petrol", "diesel", "hybrid", "electric"),
  cover_type        = c("comprehensive", "third_party_fire_theft", "third_party_only"),
  overnight_parking = c("garage", "driveway", "street"),
  vehicle_use       = c("social", "commuting", "business"),
  construction_type = c("brick", "stone", "timber", "non_standard"),
  occupancy         = c("owner_occupied", "let"),
  tenure            = c("owner", "renter"),
  security_level    = c("standard", "locks", "alarm_and_locks")
)
norm <- function(x) gsub("[ -]", "_", tolower(trimws(as.character(x))))   # "non-standard" = "non_standard"

# Extra columns shown in a column's profile, if they exist in that table
extra_profile <- c(year_built = "median", driver_age = "median", policyholder_age = "median",
                   sum_insured = "median", vehicle_value_group = "median", written_premium = "median")

# ---- Text-based suggestion for an undocumented code ------------------------
suggest <- function(v, docs) {
  if (is.na(v)) return("(blank)")
  initials <- sapply(strsplit(docs, "_"), \(w) paste(substr(w, 1, 1), collapse = ""))
  prefix   <- docs[startsWith(docs, v)]                          # "t" -> "timber"
  inits    <- docs[initials == v]                                # "ns" -> "non_standard"
  typo     <- if (nchar(v) >= 4) docs[adist(v, docs)[1, ] <= 2] else character(0)   # "detatched"
  out <- unique(c(prefix, inits, typo))
  if (length(out)) paste(out, collapse = " / ") else "no text match"
}

# ---- Claims per row: count and incurred ------------------------------------
row_claims <- claims |>
  group_by(row_id) |>
  summarise(n_claims = n(), incurred = sum(incurred_to_date, na.rm = TRUE), .groups = "drop")

# ---- Check every documented column in every line ---------------------------
found <- FALSE
for (l in lines) {
  tr <- train[[l]] |> left_join(row_claims, by = "row_id") |>
    mutate(n_claims = coalesce(n_claims, 0L), incurred = coalesce(incurred, 0))
  te <- test[[l]]

  for (col in intersect(names(documented), names(tr))) {
    docs   <- documented[[col]]
    vals_tr <- norm(tr[[col]])
    vals_te <- norm(te[[col]])
    if (all(c(vals_tr, vals_te) %in% docs)) next
    found <- TRUE

    extras <- intersect(names(extra_profile), setdiff(names(tr), col))
    prof <- tr |>
      mutate(value = vals_tr) |>
      group_by(value) |>
      summarise(train_rows  = n(),
                exposure    = round(sum(exposure)),
                claim_freq  = round(sum(n_claims) / sum(exposure), 4),
                avg_cost    = round(sum(incurred) / pmax(sum(n_claims), 1)),
                across(all_of(extras), \(x) round(median(x, na.rm = TRUE), 1), .names = "med_{.col}"),
                .groups = "drop") |>
      full_join(tibble(value = vals_te) |> count(value, name = "test_rows"), by = "value") |>
      mutate(across(c(train_rows, test_rows), \(x) coalesce(x, 0L)),
             documented = value %in% docs,
             suggestion = ifelse(documented, "", sapply(value, suggest, docs = docs))) |>
      arrange(desc(documented), value) |>
      relocate(value, documented, suggestion, train_rows, test_rows)

    cat("\n================", l, "-", col, "================\n")
    cat("Documented:", paste(docs, collapse = ", "), "\n")
    print(as.data.frame(prof), row.names = FALSE)

    # Raw values before any lowercasing, from the original file
    for (src in c(file.path(raw_dir, paste0("policies_", l, ".parquet")),
                  file.path(raw_dir, "customers.parquet"), file.path(raw_dir, "addresses.parquet"))) {
      if (!is.na(raw_dir) && file.exists(src)) {
        cols <- names(read_parquet(src, as_data_frame = FALSE))
        if (col %in% cols) {
          raw <- read_parquet(src, col_select = all_of(col))[[1]]
          cat("Raw values in", basename(src), ":\n"); print(table(raw, useNA = "ifany"))
          break
        }
      }
    }
  }
}
if (!found) cat("Every documented column has only documented values in all lines.\n")

cat("\nNote: region, credit_band and other customer/address columns repeat in each line's table;",
    "the same code should behave the same way in all three.\n")


================ motor - marital_status ================
Documented: single, married, divorced, widowed 
    value documented suggestion train_rows test_rows exposure claim_freq avg_cost med_year_built med_driver_age med_vehicle_value_group med_written_premium
 divorced       TRUE                 80417     20126    65297     0.1189     3804           1965             56                      11               456.5
  married       TRUE                452910    114806   365822     0.1267     4119           1964             55                      11               460.3
   single       TRUE                382271    103642   303941     0.1685     4095           1964             36                      11               529.9
  widowed       TRUE                 39638      9970    32128     0.1253     3919           1965             56                      12               456.5
     <NA>      FALSE    (blank)      24732      6442    19970     0.1475     4145           1964             49   

In [47]:
train_full$buildings %>%
  count(year = year(period_start), construction_type) %>%
  pivot_wider(names_from = construction_type, values_from = n, values_fill = 0)

year,brick,non_standard,stone,t,w
<dbl>,<int>,<int>,<int>,<int>,<int>
2021,109143,15406,12460,18720,0
2022,128791,18167,14632,22274,0
2023,128912,18228,14639,0,22353
2024,128798,18332,14662,0,22209


In [48]:
cm <- read_parquet(file.path(out_dir, "claims_matched.parquet")) %>%
  filter(line == "motor") %>% count(row_id, name = "n_claims")

train_full$motor %>%
  left_join(cm, by = "row_id") %>%
  mutate(n_claims = coalesce(n_claims, 0L),
         ncb_above_licence = ncb_years > years_licensed) %>%
  group_by(ncb_above_licence) %>%
  summarise(rows = n(),
            claim_freq = sum(n_claims) / sum(exposure),
            median_age = median(driver_age),
            median_licensed = median(years_licensed),
            median_ncb = median(ncb_years),
            .groups = "drop")

ncb_above_licence,rows,claim_freq,median_age,median_licensed,median_ncb
<lgl>,<int>,<dbl>,<int>,<int>,<int>
FALSE,852927,0.1368742,55,34,7
TRUE,127041,0.1838032,26,3,7


In [49]:
# =============================================================================
#
# Reads the joined tables Phase 1 saved, cleans train and test together with
# the same rules, and saves model-ready tables. Nothing here depends on
# particular years or values: fill values are calculated from each run's
# training data, and any new undocumented code stops the run with its name.
#
# Rules (evidence in the Phase 2 log):
#   - Text lowercased/trimmed; known codes mapped (t, w -> timber)
#   - Any other undocumented category value -> run stops and lists it
#   - Blank text and yes/no values -> "unknown" (blanks are random, ~2.5%)
#   - Blank numbers -> training median + <column>_missing flag
#   - telematics_score blank when not opted in -> opted-in training median,
#     with telematics_score_known flag
#   - years_licensed capped at driver_age - min_licence_age;
#     ncb_years capped at years_licensed; both flagged
#   - Derived: policy_year, customer_age, customer_tenure, property_age
#
# Output (in /kaggle/working/<out_folder>): train_<line>.parquet,
# test_<line>.parquet, cleaning_log.csv, fill_values.csv
# =============================================================================


# %% Phase 2 Cell 1: Settings ==================================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
in_folder       <- "prep"     # Phase 1 output folder (e.g. "prep_week2")
out_folder      <- "clean"    # Phase 2 output folder (e.g. "clean_week2")
lines           <- c("motor", "buildings", "contents")
unknown_label   <- "unknown"
min_licence_age <- 16         # years_licensed can't exceed driver_age minus this

# Undocumented codes and the documented value they stand for.
# Evidence: t only in 2021-22, w only in 2023-25, never "timber"; same claim frequency.
code_map <- list(
  construction_type = c(t = "timber", w = "timber")
)

# Blanks that mean something specific (the description says "if any")
na_meaning <- list(
  campaign_code = "none"
)

# Columns never used as model features
drop_cols <- c("exposure_kaggle")   # supplied exposure: contains planted errors
# ---------------------------------------------------------------------------

# Documented categories, from the dataset description
categories <- list(
  marital_status = c("single", "married", "divorced", "widowed"),
  credit_band = c("a", "b", "c", "d", "e"),
  region = sprintf("r%02d", 1:12),
  urban_density = c("rural", "suburban", "urban", "dense_urban"),
  soil_type = c("clay", "sand", "loam", "chalk", "rock"),
  property_type = c("detached", "semi_detached", "terraced", "flat"),
  roof_type = c("pitched", "flat"),
  energy_rating = c("a", "b", "c", "d", "e", "f", "g"),
  channel = c("direct", "comparison_site", "broker"),
  payment_method = c("annual", "monthly"),
  body_type = c("hatchback", "saloon", "estate", "suv", "coupe"),
  transmission = c("manual", "automatic"),
  fuel_type = c("petrol", "diesel", "hybrid", "electric"),
  cover_type = c("comprehensive", "third_party_fire_theft", "third_party_only"),
  overnight_parking = c("garage", "driveway", "street"),
  vehicle_use = c("social", "commuting", "business"),
  construction_type = c("brick", "stone", "timber", "non_standard"),
  occupancy = c("owner_occupied", "let"),
  tenure = c("owner", "renter"),
  security_level = c("standard", "locks", "alarm_and_locks")
)

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(lubridate)
})

base_dir <- if (dir.exists("/kaggle/working")) "/kaggle/working" else "."
in_dir   <- file.path(base_dir, in_folder)
out_dir2 <- file.path(base_dir, out_folder)
if (!dir.exists(in_dir)) stop("No Phase 1 output in ", in_dir, " - run the Phase 1 cells first")
dir.create(out_dir2, showWarnings = FALSE, recursive = TRUE)
cat("Phase 1 input:", in_dir, "\nPhase 2 output:", out_dir2, "\n")

Phase 1 input: /kaggle/working/prep 
Phase 2 output: /kaggle/working/clean 


In [50]:
# %% Phase 2 Cell 2: Cleaning function ========================================

norm <- function(x) gsub("[ -]+", "_", tolower(trimws(as.character(x))))
is_id   <- function(cols) grepl("_id$", cols)
is_date <- function(d) vapply(d, \(x) inherits(x, c("Date", "POSIXt")), logical(1))

clean_line <- function(tr, te, line) {
  log  <- list()
  fills <- list()
  note <- function(column, action, idx) {
    n_tr <- sum(idx & d$.set == "train", na.rm = TRUE); n_te <- sum(idx & d$.set == "test", na.rm = TRUE)
    if (n_tr + n_te > 0) log[[length(log) + 1]] <<- tibble(line = line, column = column, action = action,
                                                           train_rows = n_tr, test_rows = n_te)
  }

  d <- bind_rows(train = tr, test = te, .id = ".set") |> select(-any_of(drop_cols))
  text_cols <- names(d)[vapply(d, \(x) is.character(x) || is.factor(x), logical(1))]
  text_cols <- setdiff(text_cols[!is_id(text_cols)], ".set")

  # 1. Standardise text
  for (col in text_cols) d[[col]] <- norm(d[[col]])

  # 2. Map known undocumented codes
  for (col in intersect(names(code_map), names(d))) {
    idx <- d[[col]] %in% names(code_map[[col]])
    note(col, paste("mapped", paste(names(code_map[[col]]), code_map[[col]], sep = "->", collapse = ", ")), idx)
    d[[col]][idx] <- unname(code_map[[col]][d[[col]][idx]])
  }

  # 3. Stop on any undocumented value not covered by code_map
  for (col in intersect(names(categories), names(d))) {
    bad <- !is.na(d[[col]]) & !d[[col]] %in% categories[[col]]
    if (any(bad)) {
      tb <- table(d[[col]][bad], d$.set[bad])
      print(tb)
      stop(line, ": ", col, " has undocumented values: ", paste(rownames(tb), collapse = ", "),
           ". Check them, then add each to code_map in Phase 2 Cell 1.")
    }
  }

  # 4. Blanks with a stated meaning, then other text blanks -> unknown
  for (col in intersect(names(na_meaning), names(d))) {
    idx <- is.na(d[[col]]); note(col, paste0("blank -> \"", na_meaning[[col]], "\""), idx)
    d[[col]][idx] <- na_meaning[[col]]
  }
  for (col in text_cols) {
    idx <- is.na(d[[col]]); note(col, paste0("blank -> \"", unknown_label, "\""), idx)
    d[[col]][idx] <- unknown_label
  }

  # 5. Yes/no columns with blanks -> "true" / "false" / "unknown"
  for (col in names(d)[vapply(d, is.logical, logical(1))]) {
    if (anyNA(d[[col]])) {
      idx <- is.na(d[[col]]); note(col, paste0("yes/no blank -> \"", unknown_label, "\""), idx)
      d[[col]] <- ifelse(idx, unknown_label, ifelse(d[[col]], "true", "false"))
    }
  }

  # 6. Consistency caps (formulas, not fixed values)
  if (all(c("years_licensed", "driver_age") %in% names(d))) {
    max_lic <- pmax(d$driver_age - min_licence_age, 0)
    idx <- !is.na(d$years_licensed) & !is.na(max_lic) & d$years_licensed > max_lic
    note("years_licensed", paste("capped at driver_age -", min_licence_age), idx)
    d$years_licensed_capped <- idx
    d$years_licensed[idx] <- max_lic[idx]
  }
  if (all(c("ncb_years", "years_licensed") %in% names(d))) {
    idx <- !is.na(d$ncb_years) & !is.na(d$years_licensed) & d$ncb_years > d$years_licensed
    note("ncb_years", "capped at years_licensed", idx)
    d$ncb_capped <- idx
    d$ncb_years[idx] <- d$years_licensed[idx]
  }

  # 7. Telematics score: blank by design when not opted in
  if ("telematics_score" %in% names(d)) {
    d$telematics_score_known <- !is.na(d$telematics_score)
    fill <- median(d$telematics_score[d$.set == "train"], na.rm = TRUE)
    idx <- is.na(d$telematics_score); note("telematics_score", sprintf("blank -> opted-in training median %.1f", fill), idx)
    fills[[length(fills) + 1]] <- tibble(line = line, column = "telematics_score", fill_value = fill)
    d$telematics_score[idx] <- fill
  }

  # 8. Numeric blanks -> training median + flag (repeated after derived features)
  impute <- function(d) {
    num_cols <- names(d)[vapply(d, is.numeric, logical(1))]
    for (col in num_cols[vapply(d[num_cols], anyNA, logical(1))]) {
      fill <- median(d[[col]][d$.set == "train"], na.rm = TRUE)
      if (is.na(fill)) stop(line, ": ", col, " has no training values to fill from")
      idx <- is.na(d[[col]]); note(col, sprintf("blank -> training median %s + flag", format(fill)), idx)
      fills[[length(fills) + 1]] <<- tibble(line = line, column = col, fill_value = fill)
      d[[paste0(col, "_missing")]] <- idx
      d[[col]][idx] <- fill
    }
    d
  }
  d <- impute(d)

  # 9. Derived features from each row's own dates (after filling, so they agree
  #    with the filled inputs; a blank date of birth is filled in a second pass)
  d$policy_year <- year(d$period_start)
  if ("date_of_birth" %in% names(d))       d$customer_age    <- floor(time_length(interval(d$date_of_birth, d$period_start), "years"))
  if ("customer_since_year" %in% names(d)) d$customer_tenure <- d$policy_year - d$customer_since_year
  if ("year_built" %in% names(d))          d$property_age    <- d$policy_year - d$year_built
  d <- impute(d)

  list(train = d |> filter(.set == "train") |> select(-.set),
       test  = d |> filter(.set == "test")  |> select(-.set),
       log = bind_rows(log), fills = bind_rows(fills))
}

In [51]:
# %% Phase 2 Cell 3: Load Phase 1 output and clean each line ===================

load_in <- function(name) as_tibble(read_parquet(file.path(in_dir, paste0(name, ".parquet"))))

cleaned <- setNames(lapply(lines, \(l) {
  cat("Cleaning", l, "\n")
  clean_line(load_in(paste0("train_", l)), load_in(paste0("test_", l)), l)
}), lines)

train_clean <- lapply(cleaned, `[[`, "train")
test_clean  <- lapply(cleaned, `[[`, "test")
cleaning_log <- bind_rows(lapply(cleaned, `[[`, "log"))
fill_values  <- bind_rows(lapply(cleaned, `[[`, "fills"))

options(width = 200)
print(as.data.frame(cleaning_log), row.names = FALSE)

Cleaning motor 
Cleaning buildings 
Cleaning contents 
      line               column                                 action train_rows test_rows
     motor    overnight_parking                     blank -> "unknown"      24371      6398
     motor           occupation                     blank -> "unknown"      24158      6307
     motor       marital_status                     blank -> "unknown"      24732      6442
     motor            soil_type                     blank -> "unknown"      24741      6524
     motor            roof_type                     blank -> "unknown"      24286      6402
     motor        energy_rating                     blank -> "unknown"      24727      6425
     motor       years_licensed              capped at driver_age - 16       2962       755
     motor            ncb_years               capped at years_licensed     127106     34795
     motor     telematics_score blank -> opted-in training median 60.0     858223    221126
     motor          vehic

In [52]:
# %% Phase 2 Cell 4: Final checks and save =====================================

for (l in lines) {
  tr <- train_clean[[l]]; te <- test_clean[[l]]

  # No blanks left outside date columns
  for (s in list(list("train", tr), list("test", te))) {
    d <- s[[2]]; check <- names(d)[!is_date(d)]
    na_left <- vapply(d[check], anyNA, logical(1))
    if (any(na_left)) stop(l, " ", s[[1]], ": blanks remain in ", paste(names(na_left)[na_left], collapse = ", "))
  }
  # Only documented values (plus unknown) in category columns
  for (col in intersect(names(categories), names(tr))) {
    bad <- setdiff(unique(c(tr[[col]], te[[col]])), c(categories[[col]], unknown_label))
    if (length(bad)) stop(l, ": ", col, " still has ", paste(bad, collapse = ", "))
  }
  # Train and test: same columns, same types
  if (!identical(names(tr), names(te))) stop(l, ": train and test columns differ")
  cls_tr <- vapply(tr, \(x) class(x)[1], ""); cls_te <- vapply(te, \(x) class(x)[1], "")
  if (!identical(cls_tr, cls_te)) stop(l, ": column types differ: ", paste(names(cls_tr)[cls_tr != cls_te], collapse = ", "))
  # Row counts unchanged
  stopifnot(nrow(tr) == nrow(load_in(paste0("train_", l))), nrow(te) == nrow(load_in(paste0("test_", l))))

  cat(sprintf("  OK: %-10s train %s rows, test %s rows, %d columns, no blanks, only documented categories\n",
              l, format(nrow(tr), big.mark = ","), format(nrow(te), big.mark = ","), ncol(tr)))

  write_parquet(tr, file.path(out_dir2, paste0("train_", l, ".parquet")))
  write_parquet(te, file.path(out_dir2, paste0("test_",  l, ".parquet")))
}

write.csv(cleaning_log, file.path(out_dir2, "cleaning_log.csv"), row.names = FALSE)
write.csv(fill_values,  file.path(out_dir2, "fill_values.csv"),  row.names = FALSE)
cat("\nPhase 2 complete. Files in", out_dir2, ":\n"); print(list.files(out_dir2))

  OK: motor      train 979,968 rows, test 254,986 rows, 88 columns, no blanks, only documented categories
  OK: buildings  train 707,726 rows, test 184,155 rows, 71 columns, no blanks, only documented categories
  OK: contents   train 721,617 rows, test 187,498 rows, 72 columns, no blanks, only documented categories

Phase 2 complete. Files in /kaggle/working/clean :
[1] "cleaning_log.csv"        "fill_values.csv"         "test_buildings.parquet"  "test_contents.parquet"   "test_motor.parquet"      "train_buildings.parquet" "train_contents.parquet" 
[8] "train_motor.parquet"    


Phase 3: Target building

In [53]:
# =============================================================================
# PHASE 3: BUILDING THE TARGET 
# Target (from the brief): final cost of all claims with a loss date in the
# row's period, net of the policy excess and net of recoveries, excluding
# declined claims. Steps in this phase:
#   1. Cost of each claim (incurred)          <- this file set
#   2. Excess check (amounts already net)     <- this file set
#   3. Recoveries   4. Declined claims   5. Sum per row
#   6. Develop to ultimate   7. Trend   8. Large claims   9. Checks
#
# Reproducible: no years, dates or amounts are typed in. The valuation date
# is read from the claims file; everything else is calculated from the data.
# =============================================================================


# %% Phase 3 Cell 1: Settings ==================================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
prep_folder   <- "prep"      # Phase 1 output (matched claims)
clean_folder  <- "clean"     # Phase 2 output (cleaned policy rows)
target_folder <- "target"    # Phase 3 output
lines         <- c("motor", "buildings", "contents")
# ---------------------------------------------------------------------------

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(tidyr)
  library(lubridate)
})

base_dir   <- if (dir.exists("/kaggle/working")) "/kaggle/working" else "."
prep_dir3  <- file.path(base_dir, prep_folder)
clean_dir3 <- file.path(base_dir, clean_folder)
target_dir <- file.path(base_dir, target_folder)
for (d in c(prep_dir3, clean_dir3)) if (!dir.exists(d)) stop("Missing ", d, " - run Phases 1 and 2 first")
dir.create(target_dir, showWarnings = FALSE, recursive = TRUE)

read_out <- function(dir, name) as_tibble(read_parquet(file.path(dir, paste0(name, ".parquet"))))
fmt <- function(x, d = 0) formatC(x, format = "f", digits = d, big.mark = ",")

cat("Inputs:", prep_dir3, "and", clean_dir3, "\nOutput:", target_dir, "\n")

Inputs: /kaggle/working/prep and /kaggle/working/clean 
Output: /kaggle/working/target 


In [54]:
# %% Phase 3 Cell 2: Cost of each claim ===============================
# incurred_to_date = paid so far + current estimate for open claims.
# Only claims matched to a training row in Phase 1 are used.

claims_m <- read_out(prep_dir3, "claims_matched")

# Valuation date: the latest report date in the claims file (not typed in)
valuation_date <- max(claims_m$report_date, na.rm = TRUE)
cat("Valuation date (latest report date in the data):", format(valuation_date), "\n\n")

# Amounts must be present and sensible before anything is built on them
amount_cols <- c("paid_to_date", "incurred_to_date", "recoveries")
na_amounts <- vapply(claims_m[amount_cols], \(x) sum(is.na(x)), numeric(1))
if (any(na_amounts > 0)) stop("Blank claim amounts: ", paste(names(na_amounts)[na_amounts > 0], na_amounts[na_amounts > 0], collapse = ", "))
if (any(claims_m$incurred_to_date < 0))                           stop("Negative incurred_to_date found")
if (any(claims_m$incurred_to_date < claims_m$paid_to_date - 0.01)) stop("incurred_to_date below paid_to_date found")
if (any(claims_m$loss_date > valuation_date))                     stop("Loss dates after the valuation date found")

claim_costs <- claims_m |>
  transmute(claim_id, row_id, orig_row_id, line, peril, status,
            loss_date, report_date,
            loss_year      = year(loss_date),
            report_delay_d = as.numeric(report_date - loss_date),
            paid_to_date, incurred_to_date, recoveries,
            open_estimate  = incurred_to_date - paid_to_date,   # still to be paid on open claims
            cost_incurred  = incurred_to_date)                  # Step 1 result

# Summary by line, loss year and status
step1_summary <- claim_costs |>
  group_by(line, loss_year) |>
  summarise(claims        = n(),
            closed        = sum(status == "closed"),
            open          = sum(status == "open"),
            declined      = sum(status == "repudiated"),
            paid          = sum(paid_to_date),
            open_estimate = sum(open_estimate),
            incurred      = sum(cost_incurred),
            avg_incurred  = incurred / claims,
            .groups = "drop") |>
  mutate(open_share_of_incurred = round(open_estimate / incurred, 4))

options(width = 200)
cat("Step 1: claim cost (incurred) by line and loss year\n")
print(as.data.frame(step1_summary |> mutate(across(c(paid, open_estimate, incurred, avg_incurred), \(x) fmt(x)))), row.names = FALSE)
cat("\nClaims:", fmt(nrow(claim_costs)), "| total incurred:", fmt(sum(claim_costs$cost_incurred)), "\n")

Valuation date (latest report date in the data): 2024-12-30 

Step 1: claim cost (incurred) by line and loss year
      line loss_year claims closed open declined        paid open_estimate    incurred avg_incurred open_share_of_incurred
 buildings      2021   6195   6178   17        0  53,836,682       216,152  54,052,834        8,725                 0.0040
 buildings      2022   6792   6737   55        0  60,477,776     1,051,555  61,529,331        9,059                 0.0171
 buildings      2023   6817   6619  198        0  61,202,630     3,629,842  64,832,471        9,510                 0.0560
 buildings      2024   6571   4951 1620        0  48,423,038    18,930,911  67,353,950       10,250                 0.2811
  contents      2021   5645   5583    0       62  14,870,108             0  14,870,108        2,634                 0.0000
  contents      2022   6097   6024    0       73  16,040,969             0  16,040,969        2,631                 0.0000
  contents      2023   62

In [55]:
# %% Phase 3 Cell 3 : Excess - check, don't subtract ====================
# The description says paid_to_date is already net of excess, and incurred is
# paid + estimate. Subtracting the excess again would count it twice, so this
# step only checks the data behaves as "net of excess" would.
#
# What "net" looks like: a higher excess means fewer claims are made (small
# losses aren't worth claiming) and less is paid per policy-year. Average cost
# PER CLAIM can still rise, because the small claims drop out - so the check
# uses cost per policy-year and claim frequency, not just cost per claim.

row_excess <- bind_rows(lapply(lines, \(l)
  read_out(clean_dir3, paste0("train_", l)) |>
    select(row_id, exposure, any_of(c("voluntary_excess", "compulsory_excess"))) |>
    mutate(line = l)))
if (!all(c("voluntary_excess", "compulsory_excess") %in% names(row_excess))) stop("Excess columns missing from cleaned files")

claim_costs <- claim_costs |>
  select(-any_of(c("voluntary_excess", "compulsory_excess", "total_excess"))) |>   # safe to rerun
  left_join(row_excess |> select(-exposure, -line), by = "row_id") |>
  mutate(total_excess = voluntary_excess + compulsory_excess)
if (anyNA(claim_costs$total_excess)) stop("Some claims have no excess from their policy row")

# Check A: per policy-year, by voluntary excess (paying claims only)
row_claims_a <- claim_costs |>
  filter(status != "repudiated") |>
  group_by(row_id) |>
  summarise(n_claims = n(), cost = sum(cost_incurred), .groups = "drop")

excess_check <- row_excess |>
  left_join(row_claims_a, by = "row_id") |>
  mutate(across(c(n_claims, cost), \(x) coalesce(x, 0))) |>
  group_by(line, voluntary_excess) |>
  summarise(policy_years  = sum(exposure),
            claim_freq    = sum(n_claims) / sum(exposure),
            cost_per_year = sum(cost) / sum(exposure),
            cost_per_claim = sum(cost) / pmax(sum(n_claims), 1),
            .groups = "drop") |>
  left_join(claim_costs |> filter(status == "closed") |>
              group_by(line, voluntary_excess) |>
              summarise(smallest_1pct = quantile(cost_incurred, 0.01), .groups = "drop"),
            by = c("line", "voluntary_excess"))

# Check B: closed, paying claims smaller than their excess. If amounts were
# gross (before excess), these would be losses the customer pays in full.
below_excess <- claim_costs |>
  filter(status == "closed") |>
  group_by(line) |>
  summarise(closed_claims          = n(),
            zero_cost              = sum(cost_incurred == 0),
            positive_below_excess  = sum(cost_incurred > 0 & cost_incurred < total_excess),
            share_positive_below   = round(positive_below_excess / closed_claims, 4),
            .groups = "drop")

cat("Step 2, check A: by voluntary excess. If amounts are net, claim_freq and cost_per_year fall as excess rises,\n",
    "and the smallest closed claims sit well below the excess (a gross amount could not be below it).\n")
print(as.data.frame(excess_check |> mutate(policy_years = fmt(policy_years), claim_freq = round(claim_freq, 4),
                                           across(c(cost_per_year, cost_per_claim, smallest_1pct), \(x) fmt(x)))), row.names = FALSE)
cat("\nStep 2, check B: closed claims with cost above zero but below the row's total excess\n")
print(as.data.frame(below_excess), row.names = FALSE)

cat("\nDecision: amounts are used as supplied (already net of excess); no excess is subtracted.\n")

write_parquet(claim_costs, file.path(target_dir, "claim_costs_step2.parquet"))
write.csv(step1_summary, file.path(target_dir, "step1_summary.csv"), row.names = FALSE)
write.csv(excess_check,  file.path(target_dir, "step2_excess_check.csv"), row.names = FALSE)
cat("Saved claim_costs_step2.parquet and summaries to", target_dir, "\n")

Step 2, check A: by voluntary excess. If amounts are net, claim_freq and cost_per_year fall as excess rises,
 and the smallest closed claims sit well below the excess (a gross amount could not be below it).
      line voluntary_excess policy_years claim_freq cost_per_year cost_per_claim smallest_1pct
 buildings                0      295,675     0.0481           432          8,975            32
 buildings              250      176,999     0.0447           423          9,466            37
 buildings              500       88,705     0.0369           397         10,748            34
 buildings             1000       29,180     0.0330           343         10,389            41
  contents                0      295,160     0.0460           119          2,581            13
  contents              250      177,055     0.0388           116          2,981            14
  contents              500       89,091     0.0292            85          2,925            14
  contents             1000      

In [56]:
# %% Phase 3 Cell 4 (Step 3): Take off recoveries ==============================
# The brief asks for cost "net of recoveries": money the insurer gets back from
# other parties or from salvage is taken off each claim.
#   cost_net = incurred_to_date - recoveries
#
# Recoveries often arrive months after a claim is paid, so recent loss years
# show less recovered so far. That lag is handled in Step 6 by developing the
# NET cost, so this step only takes off what has been recovered to date.

# Use Step 2's table; if the session restarted, read it back from disk
if (!exists("claim_costs") || !"total_excess" %in% names(claim_costs)) {
  claim_costs <- read_out(target_dir, "claim_costs_step2")
}

claim_costs <- claim_costs |>
  mutate(cost_net = cost_incurred - recoveries)            # Step 3 result

# Checks: nothing negative, nothing lost
if (any(claim_costs$recoveries < 0)) stop("Negative recoveries found")
n_neg <- sum(claim_costs$cost_net < -0.01)
if (n_neg > 0) stop(n_neg, " claims would have negative net cost (recoveries above incurred) - inspect before continuing")
stopifnot(abs(sum(claim_costs$cost_incurred) - sum(claim_costs$recoveries) - sum(claim_costs$cost_net)) < 1)

# Recoveries by line and loss year: shows the size of the deduction and the lag
step3_by_year <- claim_costs |>
  group_by(line, loss_year) |>
  summarise(claims             = n(),
            claims_with_recov  = sum(recoveries > 0),
            incurred           = sum(cost_incurred),
            recoveries         = sum(recoveries),
            net                = sum(cost_net),
            .groups = "drop") |>
  mutate(recovery_rate = round(recoveries / incurred, 4))

# Recoveries by peril: shows where they come from
step3_by_peril <- claim_costs |>
  group_by(line, peril) |>
  summarise(claims            = n(),
            share_with_recov  = round(mean(recoveries > 0), 4),
            incurred          = sum(cost_incurred),
            recoveries        = sum(recoveries),
            .groups = "drop") |>
  mutate(recovery_rate = round(recoveries / incurred, 4)) |>
  arrange(line, desc(recovery_rate))

options(width = 200)
cat("Step 3: recoveries by line and loss year\n")
print(as.data.frame(step3_by_year |> mutate(across(c(incurred, recoveries, net), \(x) fmt(x)))), row.names = FALSE)
cat("\nStep 3: recoveries by peril\n")
print(as.data.frame(step3_by_peril |> mutate(across(c(incurred, recoveries), \(x) fmt(x)))), row.names = FALSE)
cat(sprintf("\nTotal incurred %s - recoveries %s = net %s (%.2f%% recovered)\n",
            fmt(sum(claim_costs$cost_incurred)), fmt(sum(claim_costs$recoveries)), fmt(sum(claim_costs$cost_net)),
            100 * sum(claim_costs$recoveries) / sum(claim_costs$cost_incurred)))

write_parquet(claim_costs, file.path(target_dir, "claim_costs_step3.parquet"))
write.csv(step3_by_year,  file.path(target_dir, "step3_recoveries_by_year.csv"),  row.names = FALSE)
write.csv(step3_by_peril, file.path(target_dir, "step3_recoveries_by_peril.csv"), row.names = FALSE)
cat("Saved claim_costs_step3.parquet and summaries to", target_dir, "\n")


Step 3: recoveries by line and loss year
      line loss_year claims claims_with_recov    incurred recoveries         net recovery_rate
 buildings      2021   6195                 0  54,052,834          0  54,052,834        0.0000
 buildings      2022   6792                 0  61,529,331          0  61,529,331        0.0000
 buildings      2023   6817                 0  64,832,471          0  64,832,471        0.0000
 buildings      2024   6571                 0  67,353,950          0  67,353,950        0.0000
  contents      2021   5645                 0  14,870,108          0  14,870,108        0.0000
  contents      2022   6097                 0  16,040,969          0  16,040,969        0.0000
  contents      2023   6235                 0  17,539,467          0  17,539,467        0.0000
  contents      2024   6075                 0  17,516,035          0  17,516,035        0.0000
     motor      2021  27711              3760 112,650,904  6,753,258 105,897,647        0.0599
     moto

In [57]:
# %% Phase 3 Cell 5: Declined claims, then cost per policy row ==
# First: the brief excludes declined (repudiated) claims, so they count as zero
#         and are not counted as claims.
# Second: add up each row's claims. Rows with no claims get zero - a real and
#         very common answer, not missing data. Totals are also kept by peril,
#         because Step 6 develops each peril separately.
# Result is the REPORTED target (as at the valuation date); Step 6 develops it.

if (!exists("claim_costs") || !"cost_net" %in% names(claim_costs)) {
  claim_costs <- read_out(target_dir, "claim_costs_step3")
}

# ---- Step 4: declined claims -------------------------------------------------
claim_costs <- claim_costs |>
  mutate(declined    = status == "repudiated",
         cost_target = if_else(declined, 0, cost_net))      # Step 4 result

step4 <- claim_costs |>
  group_by(line) |>
  summarise(claims            = n(),
            declined_incurred = sum(cost_incurred[declined]),   # amounts first: the count
            declined_net      = sum(cost_net[declined]),        # below reuses the name
            declined_claims   = sum(declined),
            .groups = "drop")
cat("Step 4: declined claims removed (cost set to zero, not counted as claims)\n")
print(as.data.frame(step4 |> mutate(across(c(declined_incurred, declined_net), \(x) fmt(x)))), row.names = FALSE)
if (any(step4$declined_net != 0)) cat("Note: some declined claims carried a cost; it is now excluded.\n")

# ---- Step 5: sum per policy row -------------------------------------------------
peril_col <- function(p) paste0("cost_", gsub("[^a-z0-9]+", "_", tolower(p)))
count_col <- function(p) paste0("n_",    gsub("[^a-z0-9]+", "_", tolower(p)))

paying <- claim_costs |> filter(!declined)

target_rows <- setNames(lapply(lines, \(l) {
  rows <- read_out(clean_dir3, paste0("train_", l)) |>
    select(row_id, orig_row_id, policy_id, period_start, period_end, policy_year, exposure)
  cl <- paying |> filter(line == l)

  # every paying claim must belong to a row in this line's training file
  orphan <- setdiff(cl$row_id, rows$row_id)
  if (length(orphan)) stop(l, ": ", length(orphan), " claims point to rows not in the cleaned file")

  totals <- cl |> group_by(row_id) |>
    summarise(n_claims = n(), cost_reported = sum(cost_target),
              open_estimate = sum(open_estimate), .groups = "drop")

  by_peril_cost <- cl |> mutate(col = peril_col(peril)) |>
    group_by(row_id, col) |> summarise(v = sum(cost_target), .groups = "drop") |>
    pivot_wider(names_from = col, values_from = v, values_fill = 0)
  by_peril_n <- cl |> mutate(col = count_col(peril)) |>
    group_by(row_id, col) |> summarise(v = n(), .groups = "drop") |>
    pivot_wider(names_from = col, values_from = v, values_fill = 0)

  out <- rows |>
    left_join(totals, by = "row_id") |>
    left_join(by_peril_cost, by = "row_id") |>
    left_join(by_peril_n, by = "row_id") |>
    mutate(across(-c(row_id, orig_row_id, policy_id, period_start, period_end, policy_year, exposure),
                  \(x) coalesce(x, 0)))

  # checks: no rows lost or added, money reconciles, nothing negative
  stopifnot(nrow(out) == nrow(rows), !anyDuplicated(out$row_id))
  if (abs(sum(out$cost_reported) - sum(cl$cost_target)) > 1) stop(l, ": row totals don't match claim totals")
  if (sum(out$n_claims) != nrow(cl)) stop(l, ": claim counts don't match")
  if (any(out$cost_reported < 0)) stop(l, ": negative row cost")
  peril_cols <- grep("^cost_", names(out), value = TRUE) |> setdiff("cost_reported")
  if (abs(sum(out[peril_cols]) - sum(out$cost_reported)) > 1) stop(l, ": peril totals don't add up")
  out
}), lines)

step5 <- bind_rows(lapply(lines, \(l) target_rows[[l]] |> mutate(line = l))) |>
  group_by(line, policy_year) |>
  summarise(rows            = n(),
            rows_with_claim = sum(n_claims > 0),
            policy_years    = sum(exposure),
            claims          = sum(n_claims),
            cost_reported   = sum(cost_reported),
            .groups = "drop") |>
  mutate(share_rows_with_claim = round(rows_with_claim / rows, 4),
         claim_freq     = round(claims / policy_years, 4),
         cost_per_year  = cost_reported / policy_years,     # reported pure premium
         cost_per_claim = cost_reported / pmax(claims, 1))

options(width = 200)
cat("\nStep 5: reported cost per policy row, by line and policy year\n")
print(as.data.frame(step5 |> mutate(across(c(policy_years, cost_reported, cost_per_year, cost_per_claim), \(x) fmt(x)))), row.names = FALSE)
cat(sprintf("\nReconciled: row totals %s = paying claims %s\n",
            fmt(sum(sapply(target_rows, \(d) sum(d$cost_reported)))), fmt(sum(paying$cost_target))))

for (l in lines) write_parquet(target_rows[[l]], file.path(target_dir, paste0("target_rows_", l, "_step5.parquet")))
write_parquet(claim_costs, file.path(target_dir, "claim_costs_step4.parquet"))
write.csv(step5, file.path(target_dir, "step5_summary.csv"), row.names = FALSE)
cat("Saved target_rows_<line>_step5.parquet and claim_costs_step4.parquet to", target_dir, "\n")

Step 4: declined claims removed (cost set to zero, not counted as claims)
      line claims declined_incurred declined_net declined_claims
 buildings  26375                 0            0               0
  contents  24052                 0            0             299
     motor 112309                 0            0            1408

Step 5: reported cost per policy row, by line and policy year
      line policy_year   rows rows_with_claim policy_years claims cost_reported share_rows_with_claim claim_freq cost_per_year cost_per_claim
 buildings        2021 155729            5982      140,783   6195    54,052,834                0.0384     0.0440           384          8,725
 buildings        2022 183864            6445      149,872   6792    61,529,331                0.0351     0.0453           411          9,059
 buildings        2023 184132            6594      149,987   6817    64,832,471                0.0358     0.0455           432          9,510
 buildings        2024 184001      

In [59]:
# %% Phase 3 Cell 6 (Step 6): Develop to ultimate ==============================
# Claims are a snapshot at the valuation date, so recent periods are missing
# claims that haven't been reported yet. This step MEASURES how late claims get
# reported, from the data, and grosses recent periods up to their expected final
# (ultimate) level. Nothing is assumed about which year or line needs it:
# a line whose claims are all reported quickly gets factors of about 1.
#
# Method: monthly chain ladder on reporting delay (loss month x months until
# reported), per line and peril. Perils with too few claims use their line's
# pooled pattern. Three versions are built side by side:
#   A  count pattern  - unreported claims assumed to cost the same as reported ones
#   B  cost pattern   - lets late-reported claims be bigger or smaller
#   C  no development - recent periods instead get a lower weight (completion)
# A one-year backtest re-runs the method as if the data had been frozen 12 months
# earlier and checks which version best predicted what was reported afterwards.
# Calibration: for each peril, the backtest compares late cost predicted vs late
# cost that actually arrived. The peril's uplift is scaled by that ratio, blended
# towards 1 by credibility (how many late claims the comparison rests on), so a
# peril judged on a handful of claims is barely changed.
#
# Limits (stated in the log): the data is one snapshot, so later changes to
# estimates on claims ALREADY reported cannot be measured; open claims are taken
# at their current estimate. Development beyond the oldest loss month observed
# is taken as complete.

# ---- Settings ---------------------------------------------------------------
min_claims_peril <- 300      # fewer claims than this -> use the line's pooled pattern
backtest_months  <- 12       # how far back the backtest freezes the data
dev_choice       <- "auto"   # "auto" = best backtest per line, or force "A", "B" or "C"
dev_calibrate    <- TRUE     # scale each peril's uplift by how well it predicted late cost in the backtest
full_cred_claims <- 1082     # late claims needed to trust a peril's backtest fully (standard credibility)
k_uplift_bounds  <- c(0.25, 2) # limits on the calibration, so one noisy backtest year can't swing a factor wildly
tail_warn_share  <- 0.02     # warn if the oldest year still has more than this share of cost in open estimates
# -----------------------------------------------------------------------------

if (!exists("claim_costs") || !"cost_target" %in% names(claim_costs)) {
  claim_costs <- read_out(target_dir, "claim_costs_step4")
}
if (!exists("valuation_date")) valuation_date <- max(claim_costs$report_date, na.rm = TRUE)

mi <- function(d) year(d) * 12 + month(d) - 1          # month index
V  <- mi(valuation_date)

cl_all <- claim_costs |>
  filter(!declined) |>
  mutate(origin = mi(loss_date), lag = pmax(mi(report_date) - origin, 0), cnt = 1)

# Cumulative development factors (age -> factor to ultimate) for one group
cdf_table <- function(d, v, val) {
  inc <- d |> group_by(origin, lag) |> summarise(x = sum(.data[[val]]), .groups = "drop")
  o_min <- min(d$origin); kmax <- v - o_min
  tri <- expand_grid(origin = o_min:v, lag = 0:kmax) |>
    filter(origin + lag <= v) |>
    left_join(inc, by = c("origin", "lag")) |>
    mutate(x = coalesce(x, 0)) |>
    arrange(origin, lag) |> group_by(origin) |> mutate(C = cumsum(x)) |> ungroup()
  f <- tri |> select(origin, lag, C) |>
    inner_join(tri |> transmute(origin, lag = lag - 1, C_next = C), by = c("origin", "lag")) |>
    group_by(lag) |>
    summarise(f = if (sum(C) > 0) sum(C_next) / sum(C) else 1, .groups = "drop") |>
    arrange(lag) |> mutate(cdf = rev(cumprod(rev(f))))
  tibble(age = 0:kmax) |> left_join(f |> select(age = lag, cdf), by = "age") |>
    mutate(cdf = coalesce(cdf, 1))
}

# Which pattern group each claim uses: its own peril, or the pooled line
peril_groups <- cl_all |> count(line, peril) |>
  mutate(group = if_else(n >= min_claims_peril, peril, "(line pooled)"))
small <- peril_groups |> filter(group == "(line pooled)")
if (nrow(small)) cat("Perils using their line's pooled pattern (fewer than", min_claims_peril, "claims):",
                     paste(small$line, small$peril, collapse = "; "), "\n")
cl_all <- cl_all |> left_join(peril_groups |> select(line, peril, group), by = c("line", "peril"))

build_cdfs <- function(cl, v) {
  bind_rows(lapply(split(cl, list(cl$line, cl$group), drop = TRUE), \(d) {
    pool <- if (d$group[1] == "(line pooled)") cl |> filter(line == d$line[1]) else d
    if (nrow(pool) < min_claims_peril) {
      warning(d$line[1], " has fewer than ", min_claims_peril, " claims - no development applied")
      return(tibble(line = d$line[1], group = d$group[1], age = 0:(v - min(cl$origin)), cdf_A = 1, cdf_B = 1))
    }
    a <- cdf_table(pool, v, "cnt"); b <- cdf_table(pool, v, "cost_target")
    tibble(line = d$line[1], group = d$group[1], age = a$age, cdf_A = a$cdf, cdf_B = b$cdf)
  }))
}

# ---- 1. Backtest: freeze the data backtest_months earlier ------------------
v_bt <- V - backtest_months
cl_bt <- cl_all |> filter(mi(report_date) <= v_bt)
backtest <- NULL
if (v_bt - min(cl_all$origin) >= backtest_months) {
  cdf_bt <- build_cdfs(cl_bt, v_bt)
  recent <- cl_all |> filter(origin > v_bt - backtest_months, origin <= v_bt) |>
    mutate(age_bt = v_bt - origin, known_then = mi(report_date) <= v_bt)
  backtest <- recent |>
    group_by(line, group, origin, age_bt) |>
    summarise(cost_then = sum(cost_target[known_then]), n_then = sum(known_then),
              cost_now = sum(cost_target), n_now = n(), .groups = "drop") |>
    left_join(cdf_bt |> rename(age_bt = age), by = c("line", "group", "age_bt")) |>
    left_join(cdf_bt |> transmute(line, group, age_bt = age - backtest_months, cdf_A_now = cdf_A, cdf_B_now = cdf_B),
              by = c("line", "group", "age_bt")) |>
    mutate(across(c(cdf_A, cdf_B, cdf_A_now, cdf_B_now), \(x) coalesce(x, 1)),
           pred_A = cost_then * cdf_A / cdf_A_now,     # predicted cost reported by now
           pred_B = cost_then * cdf_B / cdf_B_now,
           pred_C = cost_then) |>
    group_by(line) |>
    summarise(actual_now = sum(cost_now), known_then = sum(cost_then),
              pred_A = sum(pred_A), pred_B = sum(pred_B), pred_C = sum(pred_C), .groups = "drop") |>
    mutate(error_A = pred_A / actual_now - 1, error_B = pred_B / actual_now - 1, error_C = pred_C / actual_now - 1,
           best = apply(abs(cbind(A = error_A, B = error_B, C = error_C)), 1, \(e) {
             e[!is.finite(e)] <- Inf                        # line with nothing to compare
             if (all(is.infinite(e))) "A" else names(e)[which.min(e)]
           }))
  # Peril-level calibration of the uplift, credibility-weighted
  calib <- recent |>
    group_by(line, peril, group, origin, age_bt) |>
    summarise(cost_then = sum(cost_target[known_then]), cost_now = sum(cost_target),
              n_late = sum(!known_then), .groups = "drop") |>
    left_join(cdf_bt |> rename(age_bt = age), by = c("line", "group", "age_bt")) |>
    left_join(cdf_bt |> transmute(line, group, age_bt = age - backtest_months, cdf_A_now = cdf_A),
              by = c("line", "group", "age_bt")) |>
    mutate(across(c(cdf_A, cdf_A_now), \(x) coalesce(x, 1)),
           late_pred = cost_then * cdf_A / cdf_A_now - cost_then) |>
    group_by(line, peril) |>
    summarise(late_actual = sum(cost_now - cost_then), late_pred = sum(late_pred),
              n_late = sum(n_late), .groups = "drop") |>
    mutate(ratio     = if_else(late_pred > 0, late_actual / late_pred, 1),
           z         = pmin(1, sqrt(n_late / full_cred_claims)),
           k_uplift  = if (dev_calibrate) pmin(pmax(z * ratio + (1 - z), k_uplift_bounds[1]), k_uplift_bounds[2]) else 1)
  cat("\nUplift calibration per peril (k_uplift multiplies each peril's development uplift)\n")
  print(as.data.frame(calib |> mutate(across(c(late_actual, late_pred), \(x) fmt(x)),
                                      across(c(ratio, z, k_uplift), \(x) round(x, 3)))), row.names = FALSE)

  cat("\nBacktest: data frozen at", format(valuation_date %m-% months(backtest_months)),
      "- predicting cost reported since then for the", backtest_months, "loss months before it\n")
  print(as.data.frame(backtest |> mutate(across(c(actual_now, known_then, pred_A, pred_B, pred_C), \(x) fmt(x)),
                                         across(starts_with("error"), \(x) sprintf("%+.2f%%", 100 * x)))), row.names = FALSE)
} else {
  cat("\nNot enough history for a", backtest_months, "month backtest - skipped\n")
}

# ---- 2. Factors on the full data, and the version used per line ------------
cdfs <- build_cdfs(cl_all, V)
if (!exists("calib") || is.null(backtest)) calib <- tibble(line = character(), peril = character(), k_uplift = numeric())
version <- setNames(rep(if (dev_choice == "auto") "A" else dev_choice, length(lines)), lines)
if (dev_choice == "auto" && !is.null(backtest)) version[backtest$line] <- backtest$best
cat("\nVersion used per line:", paste(names(version), version, sep = " = ", collapse = ", "), "\n")

# ---- 3. Apply to claims, then to rows ---------------------------------------
cl_all <- cl_all |>
  mutate(age = V - origin) |>
  left_join(cdfs, by = c("line", "group", "age")) |>
  select(-any_of("k_uplift")) |>
  left_join(calib |> select(line, peril, k_uplift), by = c("line", "peril")) |>
  mutate(across(c(cdf_A, cdf_B), \(x) coalesce(x, 1)),
         k_uplift   = coalesce(k_uplift, 1),
         cdf_A      = 1 + (cdf_A - 1) * k_uplift,            # calibrated uplift
         cdf_B      = 1 + (cdf_B - 1) * k_uplift,
         cost_dev_A = cost_target * cdf_A,
         cost_dev_B = cost_target * cdf_B)

# line-level count pattern, for the completion weight used by version C
cdf_line <- bind_rows(lapply(lines, \(l) {
  d <- cl_all |> filter(line == l)
  if (nrow(d) == 0) return(tibble(age = 0:(V - min(cl_all$origin)), cdf = 1, line = l))   # line with no claims
  cdf_table(d, V, "cnt") |> mutate(line = l)
}))

# Tail check: development beyond the oldest loss month can't be measured, so open
# estimates there are taken as final. Warn if the oldest year is still far from settled.
tail_chk <- if ("open_estimate" %in% names(cl_all)) cl_all |> filter(origin < min(origin) + 12) |> group_by(line) |>
  summarise(open_share = sum(open_estimate) / max(sum(cost_target), 1), .groups = "drop") else tibble(line = character(), open_share = numeric())
for (i in seq_len(nrow(tail_chk))) if (tail_chk$open_share[i] > tail_warn_share)
  cat(sprintf("Note: %s - oldest loss year still has %.1f%% of cost in open estimates; development beyond the data is not measured (estimates taken as final).\n",
              tail_chk$line[i], 100 * tail_chk$open_share[i]))

for (l in lines) {
  rows <- read_out(target_dir, paste0("target_rows_", l, "_step5"))
  dev <- cl_all |> filter(line == l) |> group_by(row_id) |>
    summarise(cost_dev_A = sum(cost_dev_A), cost_dev_B = sum(cost_dev_B), .groups = "drop")
  rows <- rows |>
    select(-any_of(c("cost_dev_A", "cost_dev_B", "dev_weight", "cost_target", "dev_version"))) |>
    left_join(dev, by = "row_id") |>
    mutate(across(c(cost_dev_A, cost_dev_B), \(x) coalesce(x, 0)),
           age_mid = V - mi(period_start + (period_end - period_start) / 2)) |>
    left_join(cdf_line |> filter(line == l) |> select(age_mid = age, cdf_mid = cdf), by = "age_mid") |>
    mutate(dev_weight  = 1 / coalesce(cdf_mid, 1),                # share reported (for version C)
           dev_version = version[[l]],
           cost_target = case_when(dev_version == "A" ~ cost_dev_A,
                                   dev_version == "B" ~ cost_dev_B,
                                   TRUE               ~ cost_reported)) |>
    select(-age_mid, -cdf_mid)
  if (any(rows$cost_target < 0)) stop(l, ": negative developed cost")
  write_parquet(rows, file.path(target_dir, paste0("target_rows_", l, "_step6.parquet")))
  assign(paste0("tr6_", l), rows)
}

# ---- 4. Checks: cost per policy-year before and after, by year --------------
step6 <- bind_rows(lapply(lines, \(l) get(paste0("tr6_", l)) |> mutate(line = l))) |>
  group_by(line, policy_year) |>
  summarise(policy_years = sum(exposure),
            reported = sum(cost_reported) / sum(exposure),
            dev_A    = sum(cost_dev_A) / sum(exposure),
            dev_B    = sum(cost_dev_B) / sum(exposure),
            target   = sum(cost_target) / sum(exposure),
            min_weight = min(dev_weight),
            .groups = "drop") |>
  mutate(uplift_target = sprintf("%+.1f%%", 100 * (target / reported - 1)))

cat("\nCost per policy-year: reported vs developed (version used = target)\n")
print(as.data.frame(step6 |> mutate(across(c(policy_years, reported, dev_A, dev_B, target), \(x) fmt(x)),
                                    min_weight = round(min_weight, 3))), row.names = FALSE)

fac_print <- cdfs |> filter(age %in% c(0, 1, 3, 6, 12, 24)) |>
  pivot_wider(names_from = age, values_from = c(cdf_A, cdf_B), names_glue = "{.value}_age{age}")
cat("\nFactors to ultimate at selected ages (months since loss)\n")
print(as.data.frame(fac_print |> mutate(across(where(is.numeric), \(x) round(x, 4)))), row.names = FALSE)

write.csv(cdfs, file.path(target_dir, "step6_factors.csv"), row.names = FALSE)
write.csv(step6, file.path(target_dir, "step6_summary.csv"), row.names = FALSE)
if (!is.null(backtest)) write.csv(backtest, file.path(target_dir, "step6_backtest.csv"), row.names = FALSE)
if (nrow(calib)) write.csv(calib, file.path(target_dir, "step6_calibration.csv"), row.names = FALSE)
cat("Saved target_rows_<line>_step6.parquet, factors, summary and backtest to", target_dir, "\n")


Uplift calibration per peril (k_uplift multiplies each peril's development uplift)
      line             peril late_actual  late_pred n_late ratio     z k_uplift
 buildings accidental_damage      32,932     29,970     25 1.099 0.152    1.015
 buildings   escape_of_water     908,453    949,967    171 0.956 0.398    0.983
 buildings              fire     698,485    164,020      8 4.259 0.086    1.280
 buildings             flood     706,584    746,568     23 0.946 0.146    0.992
 buildings             storm     536,709    535,886    124 1.002 0.339    1.001
 buildings        subsidence   2,869,352  3,459,274    110 0.829 0.319    0.946
  contents accidental_damage      64,827     69,844     77 0.928 0.267    0.981
  contents   escape_of_water     261,794    262,307     94 0.998 0.295    0.999
  contents             flood     249,089    181,187     22 1.375 0.143    1.053
  contents             theft     107,189    113,008     29 0.949 0.164    0.992
     motor     bodily_injury  13,937

In [60]:
# %% Phase 3 Cell 7 (Step 7): Trend to the prediction year ======================
# Costs change over time (inflation, risk changes). This step MEASURES the yearly
# change in cost per policy-year with a Tweedie GLM, per line, and brings every
# training year to the prediction year's level.
#
#   Tweedie GLM, log link:  cost per policy-year ~ year + mix factors
#   annual trend = exp(coefficient on year) - 1
#
# - Mix-adjusted: the strongest few simple rating factors are included, so a
#   change in WHO is insured isn't mistaken for a change in prices. They are
#   picked from the data each run (largest one-way differences in cost).
# - Rows are grouped into cells before fitting. With response = cost / exposure
#   and prior weights = exposure, a Tweedie GLM gives exactly the same estimates
#   as fitting every row (its score equations only use cell totals), but fast.
# - Recent, still-developing periods count for less (weights include dev_weight).
# - Sensitivity: refitted at other Tweedie powers and without mix adjustment.
#   Year-by-year relativities are printed as a check on the straight-line trend.
# - Nothing is typed in: the prediction year comes from the test files.

# ---- Settings ---------------------------------------------------------------
tweedie_power  <- 1.5                # Tweedie power used for the applied trend
power_check    <- c(1.2, 1.5, 1.8)   # sensitivity check
mix_top_k      <- 6                  # number of mix factors to control for
mix_max_levels <- 15                 # only simple factors (few levels) are candidates
trend_rule     <- "level_then_project"
  # "level_then_project" (default): bring each year to the LATEST year's level using
  #     its own mix-adjusted year relativity, then project to the prediction year with
  #     the fitted annual trend only if the year-to-year changes all point the same
  #     way (a consistent direction); otherwise no projection beyond the latest level.
  # "estimate": straight-line trend for all years, as fitted.
  # "if_significant": straight line only if its 95% interval excludes 0.
  # "none": no trend.
# -----------------------------------------------------------------------------

if (!requireNamespace("statmod", quietly = TRUE)) install.packages("statmod")
suppressPackageStartupMessages(library(statmod))

pred_year <- max(unlist(lapply(lines, \(l) read_out(clean_dir3, paste0("test_", l))$policy_year)))
cat("Prediction year (from the test files):", pred_year, "\n")

fit_trend <- function(cells, mix, p, year_as_factor = FALSE) {
  f <- reformulate(c(if (year_as_factor) "factor(policy_year)" else "yr", mix), response = "rate")
  k <- ncol(model.matrix(f, cells))
  glm(f, family = tweedie(var.power = p, link.power = 0), data = cells, weights = w_exp,
      start = c(log(sum(cells$w_cost) / sum(cells$w_exp)), rep(0, k - 1)),
      control = glm.control(maxit = 100))
}

trend_results <- list(); year_rel <- list()
for (l in lines) {
  rows  <- read_out(target_dir, paste0("target_rows_", l, "_step6"))
  feats <- read_out(clean_dir3, paste0("train_", l))

  # candidate mix factors: text or yes/no columns with a few levels, not IDs
  cand <- names(feats)[vapply(feats, \(x) (is.character(x) || is.logical(x)) &&
                                n_distinct(x) >= 2 && n_distinct(x) <= mix_max_levels, logical(1))]
  cand <- cand[!grepl("_id$", cand)]
  d <- rows |> select(row_id, policy_year, exposure, cost_target, dev_weight) |>
    left_join(feats |> select(row_id, all_of(cand)), by = "row_id") |>
    mutate(across(all_of(cand), as.character))

  # rank candidates by exposure-weighted spread of cost per policy-year across levels
  overall <- sum(d$cost_target) / sum(d$exposure)
  spread <- vapply(cand, \(cn) {
    g <- d |> group_by(lvl = .data[[cn]]) |>
      summarise(E = sum(exposure), m = sum(cost_target) / sum(exposure), .groups = "drop")
    sum(g$E * (g$m - overall)^2) / sum(g$E)
  }, numeric(1))
  mix <- names(sort(spread, decreasing = TRUE))[seq_len(min(mix_top_k, length(spread)))]

  cells <- d |>
    group_by(policy_year, across(all_of(mix))) |>
    summarise(w_exp = sum(exposure * dev_weight), w_cost = sum(cost_target * dev_weight), .groups = "drop") |>
    filter(w_exp > 0) |>
    mutate(rate = w_cost / w_exp, yr = policy_year - pred_year)

  for (p in power_check) for (adj in c(TRUE, FALSE)) {
    m <- fit_trend(cells, if (adj) mix else character(0), p)
    b <- coef(summary(m))["yr", ]
    trend_results[[length(trend_results) + 1]] <- tibble(
      line = l, power = p, mix_adjusted = adj,
      trend = exp(b[["Estimate"]]) - 1,
      lower = exp(b[["Estimate"]] - 1.96 * b[["Std. Error"]]) - 1,
      upper = exp(b[["Estimate"]] + 1.96 * b[["Std. Error"]]) - 1,
      converged = m$converged)
  }
  cat(l, "- mix factors controlled for:", paste(mix, collapse = ", "), "\n")

  mf <- fit_trend(cells, mix, tweedie_power, year_as_factor = TRUE)
  years <- sort(unique(cells$policy_year))
  yc <- coef(mf)[paste0("factor(policy_year)", years[-1])]
  year_rel[[l]] <- tibble(line = l, policy_year = years, relativity = round(c(1, exp(yc)), 4))
}
trend_results <- bind_rows(trend_results)
year_rel <- bind_rows(year_rel)
if (!all(trend_results$converged)) warning("Some trend GLMs did not converge - check the table")

options(width = 200)
cat("\nAnnual trend in cost per policy-year (Tweedie GLM, log link), with 95% interval\n")
print(as.data.frame(trend_results |> mutate(across(c(trend, lower, upper), \(x) sprintf("%+.2f%%", 100 * x)))), row.names = FALSE)
cat("\nYear-by-year relativities to the first year (mix-adjusted, power", tweedie_power, ")\n")
print(as.data.frame(year_rel |> pivot_wider(names_from = policy_year, values_from = relativity)), row.names = FALSE)

# ---- Apply the trend --------------------------------------------------------
consistency <- year_rel |> group_by(line) |> arrange(policy_year, .by_group = TRUE) |>
  summarise(latest_year = max(policy_year),
            consistent  = { ch <- diff(log(relativity)); length(ch) > 0 && (all(ch > 0) || all(ch < 0)) },
            .groups = "drop")
applied <- trend_results |> filter(power == tweedie_power, mix_adjusted) |>
  left_join(consistency, by = "line") |>
  mutate(applied = case_when(trend_rule == "none" ~ 0,
                             trend_rule == "if_significant" & lower <= 0 & upper >= 0 ~ 0,
                             trend_rule == "level_then_project" & !consistent ~ 0,
                             TRUE ~ trend))
if (trend_rule == "level_then_project") {
  cat("\nRule: level_then_project. Each year is brought to its line's latest-year level, then projected",
      "\nto", pred_year, "with the annual trend only where year-to-year changes are consistent:\n")
  print(as.data.frame(applied |> transmute(line, latest_year, consistent_direction = consistent,
                                           fitted_trend = sprintf("%+.2f%%", 100 * trend),
                                           projection_used = sprintf("%+.2f%% a year", 100 * applied))), row.names = FALSE)
} else {
  cat("\nTrend applied (rule:", trend_rule, "):",
      paste(sprintf("%s %+.2f%% a year", applied$line, 100 * applied$applied), collapse = ", "), "\n")
}

step7 <- list()
for (l in lines) {
  t  <- applied$applied[applied$line == l]
  ly <- applied$latest_year[applied$line == l]
  rel <- year_rel |> filter(line == l) |> select(policy_year, relativity)
  rows <- read_out(target_dir, paste0("target_rows_", l, "_step6"))
  rows <- if (trend_rule == "level_then_project") {
    rel_latest <- rel$relativity[rel$policy_year == ly]
    rows |> left_join(rel, by = "policy_year") |>
      mutate(trend_factor = (rel_latest / coalesce(relativity, rel_latest)) * (1 + t)^(pred_year - ly)) |>
      select(-relativity)
  } else {
    rows |> mutate(trend_factor = (1 + t)^(pred_year - policy_year))
  }
  rows <- rows |> mutate(cost_trended = cost_target * trend_factor)
  if (any(!is.finite(rows$cost_trended)) || any(rows$cost_trended < 0)) stop(l, ": invalid trended cost")
  write_parquet(rows, file.path(target_dir, paste0("target_rows_", l, "_step7.parquet")))
  step7[[l]] <- rows |> group_by(policy_year) |>
    summarise(policy_years = sum(exposure), before = sum(cost_target) / sum(exposure),
              trend_factor = first(trend_factor), after = sum(cost_trended) / sum(exposure), .groups = "drop") |>
    mutate(line = l, .before = 1)
}
step7 <- bind_rows(step7)
cat("\nCost per policy-year before and after trending to", pred_year, "\n")
print(as.data.frame(step7 |> mutate(across(c(policy_years, before, after), \(x) fmt(x)), trend_factor = round(trend_factor, 4))), row.names = FALSE)

write.csv(trend_results, file.path(target_dir, "step7_trend.csv"), row.names = FALSE)
write.csv(year_rel,      file.path(target_dir, "step7_year_relativities.csv"), row.names = FALSE)
write.csv(step7,         file.path(target_dir, "step7_summary.csv"), row.names = FALSE)
cat("Saved target_rows_<line>_step7.parquet and trend tables to", target_dir, "\n")

Prediction year (from the test files): 2025 
motor - mix factors controlled for: marital_status, urban_density, vehicle_use, region, ncb_capped, new_business 
buildings - mix factors controlled for: flood_zone_missing, energy_rating, property_type, region, occupancy, construction_type 
contents - mix factors controlled for: tenure, region, flood_zone_missing, urban_density, credit_band, payment_method 

Annual trend in cost per policy-year (Tweedie GLM, log link), with 95% interval
      line power mix_adjusted  trend  lower   upper converged
     motor   1.2         TRUE -3.18% -6.11%  -0.17%      TRUE
     motor   1.2        FALSE -2.53% -6.64%  +1.76%      TRUE
     motor   1.5         TRUE -3.26% -6.03%  -0.41%      TRUE
     motor   1.5        FALSE -2.56% -6.68%  +1.74%      TRUE
     motor   1.8         TRUE -3.31% -5.94%  -0.61%      TRUE
     motor   1.8        FALSE -2.59% -6.71%  +1.71%      TRUE
 buildings   1.2         TRUE +8.67% +5.53% +11.90%      TRUE
 buildings   1.2 

In [61]:
# %% Phase 3 Cell 8 (Step 8): Large claims - cap and spread back ===============
# A handful of very large losses (mostly bodily injury) can make the model think
# whole groups of policies are riskier than they are. So each row's cost is capped
# at a threshold set from the data, and the amount above the cap is spread back
# across the whole line as a loading, so no money is lost:
#
#   target for the model = min(row cost, cap)
#   large_loading        = total cost / total capped cost      (per line)
#   final prediction     = model prediction x large_loading    (applied in Phase 4)
#
# The cap is a quantile of positive row costs, worked out per line each run, so it
# adapts to any dataset. Rows are used rather than single claims because the target
# is per row; rows with more than one very large claim are rare.

# ---- Settings ---------------------------------------------------------------
cap_large      <- TRUE     # FALSE = no capping (loading = 1)
cap_quantile   <- 0.995    # cap = this quantile of positive row costs, per line
# -----------------------------------------------------------------------------

step8 <- list(); large_by_year <- list()
for (l in lines) {
  rows <- read_out(target_dir, paste0("target_rows_", l, "_step7"))
  if (!"cost_trended" %in% names(rows)) stop(l, ": run Cell 7 first")
  pos <- rows$cost_trended[rows$cost_trended > 0]
  cap <- if (cap_large && length(pos)) as.numeric(quantile(pos, cap_quantile)) else Inf

  rows <- rows |>
    mutate(cost_capped = pmin(cost_trended, cap),
           above_cap   = cost_trended - cost_capped)
  loading <- if (sum(rows$cost_capped) > 0) sum(rows$cost_trended) / sum(rows$cost_capped) else 1
  rows$large_loading <- loading

  # checks: money reconciles, nothing negative
  if (abs(sum(rows$cost_capped) * loading - sum(rows$cost_trended)) > 1) stop(l, ": loading doesn't restore the total")
  if (any(rows$cost_capped < 0)) stop(l, ": negative capped cost")

  write_parquet(rows, file.path(target_dir, paste0("target_rows_", l, "_step8.parquet")))

  step8[[l]] <- tibble(line = l, cap = cap,
                       rows_capped = sum(rows$above_cap > 0),
                       share_of_rows_with_cost = sum(rows$above_cap > 0) / length(pos),
                       cost_above_cap = sum(rows$above_cap),
                       share_of_cost_above_cap = sum(rows$above_cap) / sum(rows$cost_trended),
                       large_loading = loading,
                       largest_row = max(rows$cost_trended))
  large_by_year[[l]] <- rows |> group_by(policy_year) |>
    summarise(rows_capped = sum(above_cap > 0), share_above_cap = sum(above_cap) / sum(cost_trended), .groups = "drop") |>
    mutate(line = l, .before = 1)
}
step8 <- bind_rows(step8); large_by_year <- bind_rows(large_by_year)

options(width = 200)
cat("Step 8: large claim cap per line (cap =", cap_quantile, "quantile of positive row costs)\n")
print(as.data.frame(step8 |> mutate(across(c(cap, cost_above_cap, largest_row), \(x) fmt(x)),
                                    share_of_rows_with_cost = sprintf("%.2f%%", 100 * share_of_rows_with_cost),
                                    share_of_cost_above_cap = sprintf("%.2f%%", 100 * share_of_cost_above_cap),
                                    large_loading = round(large_loading, 4))), row.names = FALSE)
cat("\nShare of cost above the cap, by year (should look random, not trending)\n")
print(as.data.frame(large_by_year |> mutate(share_above_cap = sprintf("%.2f%%", 100 * share_above_cap))), row.names = FALSE)

write.csv(step8, file.path(target_dir, "step8_large_claims.csv"), row.names = FALSE)
write.csv(large_by_year, file.path(target_dir, "step8_large_by_year.csv"), row.names = FALSE)
cat("Saved target_rows_<line>_step8.parquet to", target_dir, "\n")

Step 8: large claim cap per line (cap = 0.995 quantile of positive row costs)
      line     cap rows_capped share_of_rows_with_cost cost_above_cap share_of_cost_above_cap large_loading largest_row
     motor  54,703         491                   0.50%     35,117,511                   8.69%        1.0952  10,758,735
 buildings 194,217         127                   0.50%     13,905,966                   4.38%        1.0458     783,990
  contents  34,128         115                   0.50%      1,541,545                   2.05%        1.0209     122,692

Share of cost above the cap, by year (should look random, not trending)
      line policy_year rows_capped share_above_cap
     motor        2021         130           5.92%
     motor        2022         142           6.45%
     motor        2023         100          15.01%
     motor        2024         119           7.02%
 buildings        2021          29           4.75%
 buildings        2022          36           4.46%
 buildings  

In [62]:
# %% Phase 3 Cell 9 (Step 9): Final checks and the final target ===============
# Re-checks every step from the saved files, reconciles the money from reported
# cost to the final target, and saves one clean target file per line for Phase 4,
# plus a record of the settings and choices this run made.

final_summary <- list(); recon <- list()
for (l in lines) {
  s5 <- read_out(target_dir, paste0("target_rows_", l, "_step5"))
  s6 <- read_out(target_dir, paste0("target_rows_", l, "_step6"))
  s7 <- read_out(target_dir, paste0("target_rows_", l, "_step7"))
  s8 <- read_out(target_dir, paste0("target_rows_", l, "_step8"))
  tr <- read_out(clean_dir3, paste0("train_", l))

  # rows: same set everywhere, one per row_id, matching the cleaned training file
  ids <- list(s5$row_id, s6$row_id, s7$row_id, s8$row_id)
  if (!all(vapply(ids, \(x) setequal(x, tr$row_id) && !anyDuplicated(x), logical(1))))
    stop(l, ": target rows don't match the cleaned training rows")
  # values: finite, non-negative; exposure positive
  for (col in c("cost_reported", "cost_target", "cost_trended", "cost_capped"))
    if (any(!is.finite(s8[[col]])) || any(s8[[col]] < 0)) stop(l, ": invalid values in ", col)
  if (any(s8$exposure <= 0)) stop(l, ": rows with zero or negative exposure")
  # money: each step's total agrees with the previous step's saved total
  if (abs(sum(s5$cost_reported) - sum(s8$cost_reported)) > 1) stop(l, ": reported cost changed after Step 5")
  if (abs(sum(s6$cost_target)   - sum(s8$cost_target))   > 1) stop(l, ": developed cost changed after Step 6")
  if (abs(sum(s7$cost_trended)  - sum(s8$cost_trended))  > 1) stop(l, ": trended cost changed after Step 7")
  if (abs(sum(s8$cost_capped) * s8$large_loading[1] - sum(s8$cost_trended)) > 1) stop(l, ": loading doesn't restore the total")

  recon[[l]] <- tibble(line = l,
                       reported  = sum(s8$cost_reported),
                       developed = sum(s8$cost_target),
                       trended   = sum(s8$cost_trended),
                       capped    = sum(s8$cost_capped),
                       capped_x_loading = sum(s8$cost_capped) * s8$large_loading[1])

  final <- s8 |>
    transmute(row_id, orig_row_id, policy_id, policy_year, exposure,
              target          = cost_capped,       # what the model learns
              target_uncapped = cost_trended,      # developed + trended, before the cap
              large_loading,                       # multiply predictions by this
              dev_weight,                          # completeness of the period (for version C)
              dev_version, n_claims)
  write_parquet(final, file.path(target_dir, paste0("target_final_", l, ".parquet")))

  final_summary[[l]] <- final |> group_by(policy_year) |>
    summarise(policy_years = sum(exposure), rows = n(), share_rows_with_claim = mean(n_claims > 0),
              target_per_year = sum(target) / sum(exposure),
              with_loading_per_year = sum(target) * first(large_loading) / sum(exposure), .groups = "drop") |>
    mutate(line = l, .before = 1)
}
recon <- bind_rows(recon); final_summary <- bind_rows(final_summary)

options(width = 200)
cat("Money trail from reported cost to the final target (all rows, all years)\n")
print(as.data.frame(recon |> mutate(across(-line, \(x) fmt(x)))), row.names = FALSE)
cat("\nFinal target per policy-year, by line and year (2025 cost level)\n")
print(as.data.frame(final_summary |> mutate(policy_years = fmt(policy_years),
                                            share_rows_with_claim = round(share_rows_with_claim, 4),
                                            across(c(target_per_year, with_loading_per_year), \(x) fmt(x)))), row.names = FALSE)

# Record of what this run decided, for the log and for reproducibility
has <- function(x) exists(x, envir = globalenv(), inherits = FALSE)   # objects made by earlier cells only
run_record <- list(
  valuation_date = if (has("valuation_date")) format(valuation_date) else NA,
  prediction_year = if (has("pred_year")) pred_year else NA,
  development_versions = if (has("version") && is.character(version)) paste(names(version), version, sep = "=", collapse = "; ") else NA,
  development_calibrated = if (has("dev_calibrate")) dev_calibrate else NA,
  trend_rule = if (has("trend_rule")) trend_rule else NA,
  trend_applied = if (has("applied")) paste(sprintf("%s=%+.4f", applied$line, applied$applied), collapse = "; ") else NA,
  cap_quantile = if (has("cap_large") && cap_large) cap_quantile else NA)
writeLines(paste(names(run_record), unlist(run_record), sep = ": "), file.path(target_dir, "target_run_record.txt"))
write.csv(recon, file.path(target_dir, "target_money_trail.csv"), row.names = FALSE)
write.csv(final_summary, file.path(target_dir, "target_final_summary.csv"), row.names = FALSE)

cat("\nPhase 3 complete. Final targets: target_final_<line>.parquet in", target_dir, "\n")
cat(readLines(file.path(target_dir, "target_run_record.txt")), sep = "\n")

Money trail from reported cost to the final target (all rows, all years)
      line    reported   developed     trended      capped capped_x_loading
     motor 430,595,512 444,480,646 404,139,404 369,021,893      404,139,404
 buildings 247,768,586 255,022,201 317,509,334 303,603,368      317,509,334
  contents  65,966,579  66,653,295  75,263,046  73,721,500       75,263,046

Final target per policy-year, by line and year (2025 cost level)
      line policy_year policy_years   rows share_rows_with_claim target_per_year with_loading_per_year
     motor        2021      187,992 216008                0.1114             475                   521
     motor        2022      199,699 254685                0.1016             478                   524
     motor        2023      199,682 254577                0.0979             446                   488
     motor        2024      199,786 254698                0.0912             477                   522
 buildings        2021      140,783 155729

Phase 4 

In [83]:
# =============================================================================
# PHASE 4: TWEEDIE GLM, VALIDATION AND SUBMISSION (same notebook, after Phase 3)
#
# Competition metric: Tweedie D^2 at power 1.5 (higher is better). So each line
# gets a Tweedie GLM, power 1.5, log link, fitted to row cost with offset
# log(exposure) - which minimises exactly the deviance the metric uses.
#
# Everything is built from each run's data: inflation indexes for money features,
# bands for continuous features, which category levels are kept, base levels.
# Nothing is typed in for this dataset; for Week 2 change only the folders below.
# =============================================================================


# %% Phase 4 Cell 1: Settings and feature preparation ==========================

# Phase 4 reads everything it needs from the saved files, so clear Phases 1-3's
# objects from memory first: the GLM fits need the room.
rm(list = ls()); invisible(gc())

# ---- SETTINGS ---------------------------------------------------------------
clean_folder4   <- "clean"      # Phase 2 output
target_folder4  <- "target"     # Phase 3 output
model_folder    <- "model"      # Phase 4 output
lines           <- c("motor", "buildings", "contents")
tweedie_p       <- 1.5          # = the competition metric's power
n_bands         <- 4            # bands for continuous features (training quantiles); lower = faster, less memory
max_as_factor   <- 12           # numeric columns with this many values or fewer are used as categories
min_level_expo  <- 500          # category levels with less training exposure join "other"
min_level_claims <- 50          # ...and levels with fewer claims than this (too thin to estimate safely)
max_levels      <- 10           # at most this many levels kept per category (largest exposure first)
money_features  <- c("sum_insured", "written_premium")   # indexed to the prediction year
drop_features   <- c("customer_since_year", "year_built",   # duplicated by customer_tenure / property_age
                     "customer_service_contacts")          # leaks the claim: caused by it, not known at pricing time
max_iter        <- 25           # GLM iterations; each one is printed so progress is visible
dup_cor         <- 0.98         # numeric features this correlated with an earlier one are dropped as duplicates
max_relativity  <- 20           # review gate: any coefficient implying a relativity above this (or below 1/this) is flagged
# Review gate (human escalation): the submission is only written if every line's
# holdout results are within these limits, or a person has reviewed them and set
# reviewed_ok <- TRUE.
tol_ae          <- 0.05         # |actual / expected - 1| on the holdout year
min_d2          <- 0            # holdout D^2 must exceed this
reviewed_ok     <- FALSE
# -----------------------------------------------------------------------------

if (!requireNamespace("statmod", quietly = TRUE)) install.packages("statmod")
suppressPackageStartupMessages({ library(arrow); library(dplyr); library(tidyr); library(statmod) })

base4      <- if (dir.exists("/kaggle/working")) "/kaggle/working" else "."
clean_dir4 <- file.path(base4, clean_folder4)
tgt_dir4   <- file.path(base4, target_folder4)
model_dir  <- file.path(base4, model_folder)
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
rd <- function(dir, name) as_tibble(read_parquet(file.path(dir, paste0(name, ".parquet"))))

# Inputs from Phases 2 and 3 must exist on disk (they are lost if the session ended)
need <- c(file.path(clean_dir4, paste0(rep(c("train_", "test_"), each = length(lines)), lines, ".parquet")),
          file.path(tgt_dir4, paste0("target_final_", lines, ".parquet")))
missing_in <- need[!file.exists(need)]
if (length(missing_in)) stop("Missing inputs - run Phases 1-3 first (Run All):\n  ", paste(basename(missing_in), collapse = ", "))
phase4_ready <- TRUE
fmt4 <- function(x, d = 0) formatC(x, format = "f", digits = d, big.mark = ",")

# Tweedie deviance and D^2 (as the metric: constant-rate baseline = actual rate x exposure)
tw_dev <- function(y, mu, p = tweedie_p) {
  mu <- pmax(mu, 1e-10)
  2 * sum(ifelse(y > 0, y^(2 - p) / ((1 - p) * (2 - p)), 0) - y * mu^(1 - p) / (1 - p) + mu^(2 - p) / (2 - p))
}
d2 <- function(y, mu, expo) 1 - tw_dev(y, mu) / tw_dev(y, sum(y) / sum(expo) * expo)

# Load one line: training rows with target, and test rows, sharing feature columns
load_line <- function(l) {
  tgt <- rd(tgt_dir4, paste0("target_final_", l))
  tr  <- rd(clean_dir4, paste0("train_", l)) |> select(-any_of(c("exposure", "policy_year", "orig_row_id", "policy_id")))
  te  <- rd(clean_dir4, paste0("test_", l))
  train <- tgt |> select(row_id, orig_row_id, policy_year, exposure, target, target_uncapped, large_loading) |>
    inner_join(tr, by = "row_id")
  stopifnot(nrow(train) == nrow(tgt))
  list(train = train, test = te)
}

# Which columns are features (everything except IDs, dates, targets, bookkeeping)
feature_cols <- function(d) {
  excl <- c("row_id", "orig_row_id", "policy_id", "exposure", "policy_year", "weather_year", "exposure_kaggle",
            "target", "target_uncapped", "large_loading", drop_features)
  keep <- setdiff(names(d), excl)
  keep <- keep[!grepl("_id$", keep)]
  keep[!vapply(d[keep], \(x) inherits(x, c("Date", "POSIXt")), logical(1))]
}

# Build the preparation spec from FIT data only (bands, levels, money indexes)
make_spec <- function(fit, all_years, pred_year) {
  feats <- feature_cols(fit)
  spec <- list(feats = character(), type = list(), breaks = list(), levels = list(), base = list(), has_other = list(), money = list())
  for (m in intersect(money_features, feats)) {               # median by year, scaled to the prediction year
    med <- all_years |> group_by(policy_year) |> summarise(v = median(.data[[m]], na.rm = TRUE), .groups = "drop")
    target_v <- med$v[med$policy_year == pred_year]
    if (length(target_v) == 1 && is.finite(target_v) && target_v > 0)
      spec$money[[m]] <- setNames(target_v / med$v, med$policy_year)
  }
  fit <- index_money(fit, spec)

  # Near-duplicate numeric features (e.g. customer_age vs driver_age) make the GLM
  # unstable: banded separately, they get huge opposite coefficients that explode
  # wherever the two bands disagree. Keep the first of each near-identical pair.
  num <- feats[vapply(fit[feats], \(x) is.numeric(x) && n_distinct(x) > 2, logical(1))]
  if (length(num) > 1) {
    idx <- if (nrow(fit) > 200000) unique(as.integer(round(seq(1, nrow(fit), length.out = 200000)))) else seq_len(nrow(fit))
    cm <- suppressWarnings(abs(cor(as.matrix(fit[idx, num]), use = "pairwise.complete.obs")))
    cm[is.na(cm)] <- 0
    drop_dup <- character()
    for (i in seq_along(num)[-1]) for (j in seq_len(i - 1))
      if (!num[j] %in% drop_dup && cm[i, j] > dup_cor) { drop_dup <- c(drop_dup, num[i]); break }
    if (length(drop_dup)) {
      cat("  dropped as near-duplicates:", paste(drop_dup, collapse = ", "), "\n")
      feats <- setdiff(feats, drop_dup)
    }
    spec$dup_dropped <- drop_dup
  }

  for (f in feats) {
    x <- fit[[f]]
    if (is.numeric(x) && n_distinct(x) > max_as_factor) {
      br <- unique(quantile(x, probs = seq(0, 1, length.out = n_bands + 1)[-c(1, n_bands + 1)], na.rm = TRUE, names = FALSE))
      spec$type[[f]] <- "band"; spec$breaks[[f]] <- br
      lv <- band(x, br)
    } else {
      spec$type[[f]] <- "cat"
      lv <- as.character(x)
    }
    ex <- tapply(fit$exposure, lv, sum)
    nc <- tapply(fit$target > 0, lv, sum)
    ex <- sort(ex[ex >= min_level_expo & nc[names(ex)] >= min_level_claims], decreasing = TRUE)
    kept <- names(ex)[seq_len(min(max_levels, length(ex)))]
    if (length(kept) < 2) next                                 # no usable variation
    spec$feats <- c(spec$feats, f)
    spec$levels[[f]] <- kept
    spec$base[[f]] <- kept[1]                                   # largest exposure = base level
    spec$has_other[[f]] <- any(is.na(lv) | !lv %in% kept)       # did training have an "other" group?
  }
  spec
}
band <- function(x, br) paste0("b", findInterval(x, br))
index_money <- function(d, spec) {
  for (m in names(spec$money)) {
    k <- spec$money[[m]][as.character(d$policy_year)]
    d[[m]] <- d[[m]] * ifelse(is.na(k), 1, k)
  }
  d
}
apply_spec <- function(d, spec) {
  d <- index_money(d, spec)
  out <- tibble(.rows = nrow(d))
  for (f in spec$feats) {
    lv <- if (spec$type[[f]] == "band") band(d[[f]], spec$breaks[[f]]) else as.character(d[[f]])
    # values not kept go to "other" if training had one, otherwise to the base level
    fallback <- if (isTRUE(spec$has_other[[f]])) "other" else spec$base[[f]]
    lv[is.na(lv) | !lv %in% spec$levels[[f]]] <- fallback
    lvls <- c(spec$base[[f]], setdiff(c(spec$levels[[f]], if (isTRUE(spec$has_other[[f]])) "other"), spec$base[[f]]))
    out[[f]] <- factor(lv, levels = lvls)
  }
  out
}

fit_glm <- function(X, y, expo) {
  dat <- X; dat$.y <- y; dat$.off <- log(expo)
  f <- as.formula(paste(".y ~", paste(c(sprintf("`%s`", names(X)), "offset(.off)"), collapse = " + ")))
  mu0 <- sum(y) / sum(expo) * expo
  cat(sprintf("  fitting %s rows x %d features (each iteration prints a deviance)...\n", format(nrow(X), big.mark = ","), ncol(X)))
  flush.console()
  m <- glm(f, data = dat, mustart = pmax(mu0, 1e-8), model = FALSE, x = FALSE, y = FALSE,
           family = tweedie(var.power = tweedie_p, link.power = 0), control = glm.control(maxit = max_iter, trace = TRUE))
  rm(dat); slim_glm(m)
}
# Keep only what prediction needs; the full fit holds several copies of the data
slim_glm <- function(m) {
  m$qr <- list(pivot = m$qr$pivot, rank = m$qr$rank)
  for (k in c("residuals", "fitted.values", "effects", "R", "linear.predictors", "weights",
              "prior.weights", "data", "y", "model", "offset")) m[[k]] <- NULL
  invisible(gc()); m
}
# Predictions computed directly from the coefficients: exp(X b + log(exposure)).
# Coefficients R couldn't estimate (exact duplicates of other columns, shown as NA)
# contribute nothing, which is what standard prediction does too.
predict_glm <- function(m, X, expo) {
  X$.off <- log(expo)
  tt <- delete.response(terms(m))
  mf <- model.frame(tt, X, xlev = m$xlevels, na.action = na.pass)
  mm <- model.matrix(tt, mf, contrasts.arg = m$contrasts)
  b  <- coef(m); b[is.na(b)] <- 0
  eta <- drop(mm[, names(b), drop = FALSE] %*% b) + model.offset(mf)
  as.numeric(exp(eta))
}
cat("Phase 4 setup done. Model folder:", model_dir, "\n")

Phase 4 setup done. Model folder: /kaggle/working/model 


In [80]:
# Check: target leakage (run after Phase 4 Cell 1, before Cell 2; read-only, works for any dataset)
# A feature "leaks" if it is only known because a claim happened (e.g. it was
# recorded during or after the policy period). Signs: it separates claim rows from
# no-claim rows almost perfectly in training, and its distribution in test differs
# from training because test claims haven't happened yet.

leak <- list()
for (l in lines) {
  d  <- load_line(l)
  # include columns already dropped, so the evidence for dropping them stays visible
  fc <- union(feature_cols(d$train), intersect(drop_features, names(d$train)))
  has_claim <- d$train$target > 0
  for (f in fc) {
    x_tr <- d$train[[f]]; x_te <- d$test[[f]]
    num <- is.numeric(x_tr) || is.logical(x_tr)
    xn  <- if (num) as.numeric(x_tr) else NA
    # share of claim rows among rows where the feature is at its most common value
    top <- names(sort(table(as.character(x_tr)), decreasing = TRUE))[1]
    at_top <- as.character(x_tr) == top
    leak[[length(leak) + 1]] <- tibble(
      line = l, feature = f, dropped = f %in% drop_features,
      cor_with_claim   = if (num) suppressWarnings(cor(xn, has_claim, use = "complete.obs")) else NA_real_,
      claim_share_at_most_common = mean(has_claim[at_top]),
      claim_share_elsewhere      = mean(has_claim[!at_top]),
      most_common_value = top,
      share_most_common_train = mean(at_top),
      share_most_common_test  = mean(as.character(x_te) == top, na.rm = TRUE))
  }
  rm(d); invisible(gc())
}
leak <- bind_rows(leak) |>
  mutate(separation = if_else(share_most_common_train >= 0.05,       # ignore near-unique values
                              abs(claim_share_elsewhere - claim_share_at_most_common), 0),
         train_test_gap = abs(share_most_common_train - share_most_common_test),
         suspicion = pmax(abs(coalesce(cor_with_claim, 0)), separation)) |>
  arrange(line, desc(suspicion))

options(width = 200)
cat("Top features per line by how strongly they separate claim rows (suspects first)\n")
print(as.data.frame(leak |> group_by(line) |> slice_head(n = 6) |> ungroup() |>
                      mutate(across(where(is.double), \(x) round(x, 4)))), row.names = FALSE)
write.csv(leak, file.path(model_dir, "leakage_check.csv"), row.names = FALSE)

Top features per line by how strongly they separate claim rows (suspects first)
      line                   feature dropped cor_with_claim claim_share_at_most_common claim_share_elsewhere most_common_value share_most_common_train share_most_common_test separation train_test_gap
 buildings customer_service_contacts    TRUE         0.5639                     0.0001                0.1003                 0                  0.6436                 0.6688     0.1002         0.0252
 buildings           written_premium   FALSE         0.0483                     0.0323                0.0358            183.71                  0.0001                 0.0000     0.0000         0.0001
 buildings                year_built    TRUE        -0.0447                     0.0350                0.0359              1900                  0.0328                 0.0330     0.0000         0.0002
 buildings              property_age   FALSE         0.0445                     0.0291                0.0359            

In [84]:
# %% Phase 4 Cell 2: Validation - fit on earlier years, test on the latest =====
# The latest training year is held out. Each line's model is fitted on the other
# years (bands, levels, indexes and duplicate checks built only from them) and
# scored on the held-out year with the competition's metric, against baselines.
#
# Self-correcting and self-documenting, so it does not need editing per dataset:
# - Unstable fits: if any coefficient implies a relativity beyond unstable_relativity
#   (default 100x - a numerical blow-up, not a real risk effect), the feature(s)
#   responsible are dropped and the line is refitted (up to max_refits times).
#   Large but possible effects (beyond max_relativity, default 20x) are not dropped
#   automatically; they are flagged for human review by the gate. Every drop is printed and saved to model/auto_dropped_<line>.txt,
#   and Cell 3 applies the same drops to the final model.
# - A line that fails outright (error, memory) is reported and the others continue.
# - The review gate (convergence, A/E, D^2, relativities) is saved to
#   model/review_gate.txt; Cell 3 will not write a submission unless it passed or
#   a person signed off with reviewed_ok <- TRUE in Cell 1.

if (!exists("phase4_ready")) stop("Run Phase 4 Cell 1 first (it defines the settings and functions)")
if (!exists("max_relativity")) max_relativity <- 20     # defaults if an older Cell 1 is in use
if (!exists("max_refits"))     max_refits     <- 3
if (!exists("unstable_relativity")) unstable_relativity <- 100

# Fit a line; drop features that produce extreme relativities and refit
fit_stable <- function(fit_d, all_years, pred_year, l, extra_drop = character()) {
  base_drop <- drop_features
  on.exit(drop_features <<- base_drop, add = TRUE)
  for (attempt in 0:max_refits) {
    drop_features <<- unique(c(base_drop, extra_drop))
    spec <- make_spec(fit_d, all_years, pred_year)
    m <- fit_glm(apply_spec(fit_d, spec), fit_d$target, fit_d$exposure)
    cf <- coef(m)[-1]; cf <- cf[!is.na(cf)]
    bad <- names(cf)[abs(cf) > log(unstable_relativity)]
    if (!length(bad) || attempt == max_refits) break
    # map coefficient names (feature + level) back to features: longest matching name
    culprits <- unique(vapply(bad, \(t) {
      hits <- spec$feats[startsWith(t, spec$feats) | startsWith(t, paste0("`", spec$feats))]
      if (length(hits)) hits[which.max(nchar(hits))] else NA_character_
    }, character(1)))
    culprits <- culprits[!is.na(culprits)]
    if (!length(culprits)) break
    cat(sprintf("   unstable: %s -> dropping %s and refitting\n",
                paste(sprintf("%s = %.3g", bad, exp(coef(m)[bad])), collapse = "; "), paste(culprits, collapse = ", ")))
    extra_drop <- c(extra_drop, culprits)
    rm(m); invisible(gc())
  }
  list(m = m, spec = spec, extra_drop = extra_drop)
}

invisible(gc())
holdout <- list(); lift <- list(); fails <- character()
for (l in lines) {
  cat("\n==>", l, "- loading data\n"); flush.console()
  res <- tryCatch({
    d <- load_line(l)
    pred_year <- max(d$test$policy_year)
    ho_year   <- max(d$train$policy_year)
    fit_d <- d$train |> filter(policy_year < ho_year)
    ho_d  <- d$train |> filter(policy_year == ho_year)
    all_years <- bind_rows(d$train |> select(policy_year, any_of(money_features)),
                           d$test  |> select(policy_year, any_of(money_features)))
    rm(d); invisible(gc())

    t0 <- Sys.time()
    st <- fit_stable(fit_d, all_years, pred_year, l)
    m <- st$m; spec <- st$spec
    writeLines(st$extra_drop, file.path(model_dir, paste0("auto_dropped_", l, ".txt")))
    cat(sprintf("%-10s fitted on %s rows, %d features, %d coefficients, converged: %s, %.1f min%s\n",
                l, fmt4(nrow(fit_d)), length(spec$feats), length(coef(m)), m$converged,
                as.numeric(difftime(Sys.time(), t0, units = "mins")),
                if (length(st$extra_drop)) paste0(" (auto-dropped: ", paste(st$extra_drop, collapse = ", "), ")") else ""))

    pred <- predict_glm(m, apply_spec(ho_d, spec), ho_d$exposure) * ho_d$large_loading
    actual <- ho_d$target_uncapped

    # baselines: constant rate from the fit years; premium x loss ratio if premium exists
    rate <- sum(fit_d$target_uncapped) / sum(fit_d$exposure)
    base_rate <- rate * ho_d$exposure
    prem <- if ("written_premium" %in% names(ho_d)) {
      lr <- sum(fit_d$target_uncapped) / sum(fit_d$written_premium); ho_d$written_premium * lr } else NA

    # diagnostics: largest effects and the spread of predicted rates
    cf <- coef(m)[-1]; cf <- cf[!is.na(cf)]
    big <- head(cf[order(-abs(cf))], 6)
    cat("   largest relativities:", paste(sprintf("%s = %.2f", names(big), exp(big)), collapse = "; "), "\n")
    pr <- pred / ho_d$exposure
    cat(sprintf("   predicted rate per policy-year: median %.0f, 99.9th pct %.0f, max %.0f\n",
                median(pr), quantile(pr, 0.999), max(pr)))
    write.csv(tibble(term = names(coef(m)), coefficient = coef(m), relativity = exp(coef(m))),
              file.path(model_dir, paste0("validation_coefs_", l, ".csv")), row.names = FALSE)

    s_model <- d2(actual, pred, ho_d$exposure)
    s_rate  <- d2(actual, base_rate, ho_d$exposure)
    s_prem  <- if (all(is.na(prem))) NA_real_ else d2(actual, prem, ho_d$exposure)
    h <- tibble(line = l, holdout_year = ho_year, rows = nrow(ho_d),
                actual_total = sum(actual), predicted_total = sum(pred),
                a_over_e = sum(actual) / sum(pred),
                d2_model = s_model, d2_const_rate_fit_years = s_rate, d2_premium_x_lr = s_prem,
                max_relativity = exp(max(abs(cf))), converged = m$converged,
                auto_dropped = paste(st$extra_drop, collapse = ", "))
    dec <- ntile(pred / ho_d$exposure, 10)
    lf <- tibble(line = l, decile = dec, a = actual, e = pred, x = ho_d$exposure) |>
      group_by(line, decile) |>
      summarise(policy_years = sum(x), actual_rate = sum(a) / sum(x), predicted_rate = sum(e) / sum(x), .groups = "drop") |>
      mutate(a_over_e = actual_rate / predicted_rate)
    hp <- tibble(actual = actual, pred = pred, expo = ho_d$exposure)
    rm(m, fit_d, ho_d, st); invisible(gc())
    list(h = h, lf = lf, hp = hp)
  }, error = function(e) {
    cat("   FAILED:", conditionMessage(e), "\n"); NULL
  })

  if (is.null(res)) { fails <- c(fails, l); next }
  holdout[[l]] <- res$h; lift[[l]] <- res$lf
  assign(paste0("ho_pred_", l), res$hp)
  h <- res$h
  if (!h$converged || abs(h$a_over_e - 1) > tol_ae || h$d2_model <= min_d2 || h$max_relativity > max_relativity)
    fails <- c(fails, l)
}
holdout <- bind_rows(holdout); lift <- bind_rows(lift)

options(width = 200)
if (nrow(holdout)) {
  allp <- bind_rows(lapply(intersect(lines, holdout$line), \(l) get(paste0("ho_pred_", l))))
  cat(sprintf("\nAll lines together on the holdout year: D^2 = %.4f\n", d2(allp$actual, allp$pred, allp$expo)))
  cat("\nHoldout results per line (D^2 at power", tweedie_p, "- higher is better)\n")
  print(as.data.frame(holdout |> mutate(across(c(actual_total, predicted_total), \(x) fmt4(x)),
                                        across(c(a_over_e, starts_with("d2"), max_relativity), \(x) round(x, 4)))), row.names = FALSE)
  cat("\nLift by predicted-rate decile (actual should rise with predicted, a_over_e near 1)\n")
  print(as.data.frame(lift |> mutate(across(c(actual_rate, predicted_rate), \(x) round(x, 1)),
                                     a_over_e = round(a_over_e, 3), policy_years = fmt4(policy_years))), row.names = FALSE)
  write.csv(holdout, file.path(model_dir, "validation_holdout.csv"), row.names = FALSE)
  write.csv(lift,    file.path(model_dir, "validation_lift.csv"),    row.names = FALSE)
}

# ---- Review gate --------------------------------------------------------------
fails <- unique(fails)
if (length(fails)) {
  cat("\nREVIEW NEEDED for:", paste(fails, collapse = ", "),
      "\n(failed, not converged, holdout A/E outside +/-", tol_ae, ", D^2 not above", min_d2,
      ", or a relativity beyond", max_relativity, "x)\n")
  if (!reviewed_ok) cat("Cell 3 will not write a submission until a person reviews these results and sets reviewed_ok <- TRUE in Cell 1.\n")
} else cat("\nAll lines pass the review gate.\n")
gate_pass <- length(fails) == 0 || reviewed_ok
writeLines(c(paste("lines_failing:", paste(fails, collapse = ",")),
             paste("reviewed_ok:", reviewed_ok),
             paste("gate_pass:", gate_pass),
             paste("validated_at:", format(Sys.time(), "%Y-%m-%d %H:%M:%S")),
             paste("targets_modified:", format(max(file.mtime(file.path(tgt_dir4, paste0("target_final_", lines, ".parquet")))), "%Y-%m-%d %H:%M:%S"))),
           file.path(model_dir, "review_gate.txt"))
rm(list = ls(pattern = "^ho_pred_")); invisible(gc())


==> motor - loading data
  dropped as near-duplicates: years_licensed, customer_age, customer_tenure 
  fitting 725,270 rows x 69 features (each iteration prints a deviance)...
Deviance = 76964253 Iterations - 1
Deviance = 76213164 Iterations - 2
Deviance = 76194062 Iterations - 3
Deviance = 76194014 Iterations - 4
Deviance = 76194013 Iterations - 5
Deviance = 80871928 Iterations - 1
Deviance = 80871799 Iterations - 2
Deviance = 80871799 Iterations - 3
motor      fitted on 725,270 rows, 69 features, 209 coefficients, converged: TRUE, 2.5 min
   largest relativities: bedroomsother = 0.40; prior_claims_5yother = 1.86; prior_claims_5y3 = 1.82; ncb_yearsb0 = 1.82; prior_claims_5y2 = 1.55; voluntary_excess1000 = 0.70 
   predicted rate per policy-year: median 406, 99.9th pct 2695, max 6142

==> buildings - loading data
  dropped as near-duplicates: customer_age, customer_tenure 
  fitting 523,725 rows x 53 features (each iteration prints a deviance)...
Deviance = 73780711 Iterations - 1
De

In [85]:
# %% Phase 4 Cell 3: Final fit on all years, predict, submit ===================
# Refits each line on every training year, predicts each test row
# (model rate x exposure x large-claim loading), combines rows split across
# years, checks the file against sample_submission, and writes submission.csv
# only if the review gate passed.

if (!exists("phase4_ready")) stop("Run Phase 4 Cell 1 first (it defines the settings and functions)")
# Review gate: read the saved decision from Cell 2 (works after a restart too)
gate_file <- file.path(model_dir, "review_gate.txt")
if (!file.exists(gate_file)) stop("No validation found - run Phase 4 Cell 2 first")
gate <- setNames(sub("^[^:]+: ?", "", readLines(gate_file)), sub(":.*", "", readLines(gate_file)))
targets_now <- format(max(file.mtime(file.path(tgt_dir4, paste0("target_final_", lines, ".parquet")))), "%Y-%m-%d %H:%M:%S")
if (gate[["targets_modified"]] != targets_now) stop("Targets changed since validation - rerun Phase 4 Cell 2 first")
if (gate[["gate_pass"]] != "TRUE" && !reviewed_ok)
  stop("Review gate not passed for: ", gate[["lines_failing"]],
       ". Review Cell 2's results; if acceptable set reviewed_ok <- TRUE in Cell 1 and rerun Cells 1-3")
if (nzchar(gate[["lines_failing"]])) cat("Review gate overridden by a person (reviewed_ok = TRUE) for:", gate[["lines_failing"]], "\n")
invisible(gc())

preds <- list(); relativities <- list(); fit_info <- list()
for (l in lines) {
  cat("\n==>", l, "- loading data\n"); flush.console()
  d <- load_line(l)
  pred_year <- max(d$test$policy_year)
  all_years <- bind_rows(d$train |> select(policy_year, any_of(money_features)),
                         d$test  |> select(policy_year, any_of(money_features)))
  # apply the same automatic drops as validation (Cell 2), then the same stability check
  ad_file <- file.path(model_dir, paste0("auto_dropped_", l, ".txt"))
  auto_drop <- if (file.exists(ad_file)) readLines(ad_file) else character()
  auto_drop <- auto_drop[nzchar(auto_drop)]
  base_drop <- drop_features
  drop_features <- unique(c(base_drop, auto_drop))
  spec <- make_spec(d$train, all_years, pred_year)
  t0 <- Sys.time()
  m <- fit_glm(apply_spec(d$train, spec), d$train$target, d$train$exposure)
  drop_features <- base_drop
  cf_chk <- coef(m)[-1]; cf_chk <- cf_chk[!is.na(cf_chk)]
  if (exists("max_relativity") && any(abs(cf_chk) > log(max_relativity)) && !reviewed_ok)
    stop(l, ": final model has relativities beyond ", max_relativity, "x (",
         paste(names(cf_chk)[abs(cf_chk) > log(max_relativity)], collapse = ", "),
         "). Review, then set reviewed_ok <- TRUE or add the feature to drop_features")
  loading <- d$train$large_loading[1]

  p <- predict_glm(m, apply_spec(d$test, spec), d$test$exposure) * loading
  if (any(!is.finite(p)) || any(p < 0)) stop(l, ": invalid predictions")

  # sanity check: predicted cost per policy-year vs the training target level
  fit_info[[l]] <- tibble(line = l, rows_train = nrow(d$train), features = length(spec$feats),
                          coefficients = length(coef(m)), converged = m$converged,
                          minutes = round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1),
                          train_rate = sum(d$train$target_uncapped) / sum(d$train$exposure),
                          test_pred_rate = sum(p) / sum(d$test$exposure))
  preds[[l]] <- tibble(orig_row_id = d$test$orig_row_id, prediction = p)
  cf <- coef(m); relativities[[l]] <- tibble(line = l, term = names(cf), relativity = exp(cf))
  saveRDS(spec, file.path(model_dir, paste0("spec_", l, ".rds")))
  rm(m, d); invisible(gc())                  # free memory before the next line
}
fit_info <- bind_rows(fit_info)
options(width = 200)
cat("Final fits (test predicted cost per policy-year should be close to the training level)\n")
print(as.data.frame(fit_info |> mutate(across(c(train_rate, test_pred_rate), \(x) round(x, 1)))), row.names = FALSE)
write.csv(bind_rows(relativities), file.path(model_dir, "glm_relativities.csv"), row.names = FALSE)
write.csv(fit_info, file.path(model_dir, "final_fit_info.csv"), row.names = FALSE)

# ---- Submission ---------------------------------------------------------------
raw_hits <- unique(dirname(list.files(c("/kaggle/input", "data"), pattern = "^customers\\.parquet$", recursive = TRUE, full.names = TRUE)))
if (length(raw_hits) != 1) stop("Found ", length(raw_hits), " raw data folders - attach only one dataset or set the path")
sample_sub <- as_tibble(read_parquet(file.path(raw_hits, "sample_submission.parquet"))) |> mutate(row_id = as.character(row_id))

sub <- bind_rows(preds) |> group_by(row_id = orig_row_id) |> summarise(prediction = sum(prediction), .groups = "drop")
sub <- sample_sub |> select(row_id) |> left_join(sub, by = "row_id")
if (nrow(sub) != nrow(sample_sub))            stop("Row count differs from sample_submission")
if (anyNA(sub$prediction))                    stop(sum(is.na(sub$prediction)), " rows without a prediction")
if (any(sub$prediction < 0))                  stop("Negative predictions")
if (!setequal(sub$row_id, sample_sub$row_id)) stop("row_ids differ from sample_submission")

write.csv(sub, file.path(base4, "submission.csv"), row.names = FALSE)
cat(sprintf("\nSaved submission.csv: %s rows, mean prediction %.2f\n", fmt4(nrow(sub)), mean(sub$prediction)))
print(head(sub, 3))


==> motor - loading data
  dropped as near-duplicates: years_licensed, customer_age, customer_tenure 
  fitting 979,968 rows x 69 features (each iteration prints a deviance)...
Deviance = 104204325 Iterations - 1
Deviance = 103237974 Iterations - 2
Deviance = 103213576 Iterations - 3
Deviance = 103213511 Iterations - 4
Deviance = 103213510 Iterations - 5
Deviance = 109350349 Iterations - 1
Deviance = 109350203 Iterations - 2
Deviance = 109350203 Iterations - 3

==> buildings - loading data
  dropped as near-duplicates: customer_age, customer_tenure 
  fitting 707,726 rows x 53 features (each iteration prints a deviance)...
Deviance = 98366449 Iterations - 1
Deviance = 93152420 Iterations - 2
Deviance = 90893365 Iterations - 3
Deviance = 89950907 Iterations - 4
Deviance = 89705068 Iterations - 5
Deviance = 89683201 Iterations - 6
Deviance = 89682886 Iterations - 7
Deviance = 89682883 Iterations - 8
Deviance = 89682883 Iterations - 9
Deviance = 96818385 Iterations - 1
Deviance = 9681654